In [1]:
# ── CELL 1: Environment Setup — Mount Drive and Install GROMACS ─────────────

import os
import subprocess
from pathlib import Path

# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = Path('/content/drive/MyDrive/PfDHFR_MD')
SYSTEMS_DIR = DRIVE_BASE / 'systems'
RESULTS_DIR = DRIVE_BASE / 'md_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Drive mounted successfully.")
print(f"  Systems dir : {SYSTEMS_DIR}")
print(f"  Results dir : {RESULTS_DIR}")

# Confirm pyrimethamine zip is present
pyr_zip = SYSTEMS_DIR / 'pyrimethamine_system.zip'
assert pyr_zip.exists(), f"ERROR: zip not found at {pyr_zip}"
print(f"  pyrimethamine_system.zip : FOUND ({pyr_zip.stat().st_size / 1024:.1f} KB)")

# ── 2. Install GROMACS via miniforge ─────────────────────────────────────────
print("\nSetting up GROMACS...")

# Add miniforge to PATH first
os.environ['PATH'] = '/content/miniforge/bin:' + os.environ['PATH']

# Check if miniforge already installed
miniforge_exists = os.path.exists('/content/miniforge/bin/conda')

if not miniforge_exists:
    print("  Installing miniforge...")
    cmds = [
        "wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O miniforge.sh",
        "bash miniforge.sh -b -p /content/miniforge",
    ]
    for cmd in cmds:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if result.returncode != 0:
            print(result.stderr[-500:])
            raise RuntimeError(f"Failed: {cmd}")
else:
    print("  Miniforge already installed — skipping.")

# Check if GROMACS already installed
gmx_check = subprocess.run('gmx --version', shell=True, capture_output=True, text=True)
if gmx_check.returncode != 0:
    print("  Installing GROMACS (this takes 3-5 minutes)...")
    result = subprocess.run(
        "/content/miniforge/bin/conda install -y -c conda-forge gromacs -q",
        shell=True, capture_output=True, text=True
    )
    if result.returncode != 0:
        print(result.stderr[-500:])
        raise RuntimeError("GROMACS installation failed.")
else:
    print("  GROMACS already installed — skipping.")

# ── 3. Verify ────────────────────────────────────────────────────────────────
result = subprocess.run('gmx --version', shell=True, capture_output=True, text=True)
assert result.returncode == 0, "ERROR: gmx not callable"

for line in result.stdout.splitlines():
    if 'GROMACS version' in line:
        print(f"\n  GROMACS installed: {line.strip()}")
        break

print("\n" + "=" * 60)
print("  CELL 1 COMPLETE")
print("  Drive mounted, GROMACS ready")
print("  Next: Cell 2 — unzip system files")
print("=" * 60)

Mounted at /content/drive
Drive mounted successfully.
  Systems dir : /content/drive/MyDrive/PfDHFR_MD/systems
  Results dir : /content/drive/MyDrive/PfDHFR_MD/md_results
  pyrimethamine_system.zip : FOUND (204.4 KB)

Setting up GROMACS...
  Installing miniforge...
  Installing GROMACS (this takes 3-5 minutes)...

  GROMACS installed: GROMACS version:     2026.0-conda_forge

  CELL 1 COMPLETE
  Drive mounted, GROMACS ready
  Next: Cell 2 — unzip system files


In [ ]:
# ── CELL 2: Unzip Pyrimethamine System Files ─────────────────────────────────

import zipfile
import shutil

# ── Work directory (local Colab storage — fast I/O) ──────────────────────────
WORK_DIR = Path('/content/md_work/pyrimethamine')
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"Work directory: {WORK_DIR}")

# ── Unzip system files ────────────────────────────────────────────────────────
pyr_zip = SYSTEMS_DIR / 'pyrimethamine_system.zip'

print(f"Extracting: {pyr_zip.name} ...")
with zipfile.ZipFile(pyr_zip, 'r') as zf:
    zf.extractall(WORK_DIR)

# ── List extracted contents ───────────────────────────────────────────────────
print("\nExtracted files:")
all_files = sorted(WORK_DIR.rglob('*'))
for f in all_files:
    if f.is_file():
        print(f"  {f.relative_to(WORK_DIR)}  ({f.stat().st_size / 1024:.1f} KB)")

# ── Verify required files are present ────────────────────────────────────────
required = ['complex.gro', 'topol_complex.top', 'posre.itp']
missing = []
for req in required:
    matches = list(WORK_DIR.rglob(req))
    if not matches:
        missing.append(req)
    else:
        print(f"\n  {req} : FOUND at {matches[0].relative_to(WORK_DIR)}")

if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

print("\n" + "=" * 60)
print("  CELL 2 COMPLETE")
print("  System files extracted and verified")
print("  Next: Cell 3 — solvate (add water box)")
print("=" * 60)

Work directory: /content/md_work/pyrimethamine
Extracting: pyrimethamine_system.zip ...

Extracted files:
  complex.gro  (168.8 KB)
  em.mdp  (0.5 KB)
  md.mdp  (1.4 KB)
  npt.mdp  (1.2 KB)
  nvt.mdp  (1.3 KB)
  posre.itp  (56.2 KB)
  pyrimethamine_GMX.gro  (1.2 KB)
  pyrimethamine_GMX.itp  (15.8 KB)
  topol_complex.top  (1067.0 KB)

  complex.gro : FOUND at complex.gro

  topol_complex.top : FOUND at topol_complex.top

  posre.itp : FOUND at posre.itp

  CELL 2 COMPLETE
  System files extracted and verified
  Next: Cell 3 — solvate (add water box)


In [ ]:
# ── CELL 3: Solvate — Add Water Box ──────────────────────────────────────────

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

# ── Run gmx solvate ───────────────────────────────────────────────────────────
# -cp : input structure (protein + ligand in empty box)
# -cs : water model template (spc216 = standard TIP3P template in GROMACS)
# -o  : output structure with water added
# -p  : topology file — GROMACS will update [molecules] section automatically
print("\nRunning gmx solvate...")
result = subprocess.run(
    'gmx solvate -cp complex.gro -cs spc216.gro -o solvated.gro -p topol_complex.top',
    shell=True, capture_output=True, text=True
)

print(result.stdout[-1000:])
if result.stderr:
    print(result.stderr[-1000:])

if result.returncode != 0:
    raise RuntimeError("gmx solvate failed.")

# ── Parse water count from output ─────────────────────────────────────────────
n_water = None
for line in (result.stdout + result.stderr).splitlines():
    if 'Number of solvent molecules' in line or 'SOL' in line.upper():
        print(f"  >> {line.strip()}")
    if 'solvent molecules' in line.lower():
        try:
            n_water = int(line.strip().split()[-1])
        except ValueError:
            pass

# Verify output file created
solvated = WORK_DIR / 'solvated.gro'
assert solvated.exists(), "ERROR: solvated.gro not created"
print(f"\n  solvated.gro created ({solvated.stat().st_size / 1024:.1f} KB)")

print("\n" + "=" * 60)
print("  CELL 3 COMPLETE")
print("  Water box added successfully")
if n_water:
    print(f"  Water molecules added: {n_water:,}")
print("  Next: Cell 4 — add ions (neutralise + 0.15 M NaCl)")
print("=" * 60)

Working directory: /content/md_work/pyrimethamine

Running gmx solvate...

         based on residue and atom names, since they could not be
         definitively assigned from the information in your input
         files. These guessed numbers might deviate from the mass
         and radius of the atom type. Please check the output
         files if necessary. Note, that this functionality may
         be removed in a future GROMACS version. Please, consider
         using another file format for your input.

         The atomic radii are set according to:

++++ PLEASE READ AND CITE THE FOLLOWING REFERENCE ++++
A. Bondi
van der Waals Volumes and Radii
J. Phys. Chem. (1964)
https://doi.org/10.1021/j100785a001
-------- -------- --- Thank You --- -------- --------

Adding line for 3263 solvent molecules with resname (SOL) to topology file (topol_complex.top)


Command line:
  gmx solvate -cp complex.gro -cs spc216.gro -o solvated.gro -p topol_complex.top

Reading solute configuration
Rea

In [ ]:
# ── CELL 4 FIX 3: Increase maxwarn, run grompp + genion ──────────────────────

os.chdir(WORK_DIR)

# ── Step 4a: grompp ───────────────────────────────────────────────────────────
print("Step 4a: grompp for genion...")
result = subprocess.run(
    'gmx grompp -f em.mdp -c solvated.gro -p topol_complex.top -o ions.tpr -maxwarn 15',
    shell=True, capture_output=True, text=True
)

if result.returncode != 0:
    print(result.stdout[-500:])
    print(result.stderr[-1500:])
    raise RuntimeError("grompp for ions failed.")

assert (WORK_DIR / 'ions.tpr').exists()
print("  ions.tpr created successfully")

# ── Step 4b: genion ───────────────────────────────────────────────────────────
print("\nStep 4b: Adding ions (neutralise + 0.15 M NaCl)...")
result = subprocess.run(
    'echo "SOL" | gmx genion -s ions.tpr -o ionised.gro '
    '-p topol_complex.top -pname NA -nname CL -neutral -conc 0.15',
    shell=True, capture_output=True, text=True
)

if result.returncode != 0:
    print(result.stdout[-500:])
    print(result.stderr[-1500:])
    raise RuntimeError("gmx genion failed.")

for line in (result.stdout + result.stderr).splitlines():
    if 'Replacing' in line or 'Will try' in line or 'Adding' in line:
        print(f"  >> {line.strip()}")

ionised = WORK_DIR / 'ionised.gro'
assert ionised.exists()
print(f"\n  ionised.gro created ({ionised.stat().st_size / 1024:.1f} KB)")

print("\n" + "=" * 60)
print("  CELL 4 COMPLETE")
print("  Ions added — system ready for energy minimisation")
print("  Next: Cell 5 — energy minimisation")
print("=" * 60)

Step 4a: grompp for genion...
  ions.tpr created successfully

Step 4b: Adding ions (neutralise + 0.15 M NaCl)...
  >> Will try to add 12 NA ions and 21 CL ions.
  >> Replacing 33 solute molecules in topology file (topol_complex.top)  by 12 NA and 21 CL ions.
  >> Replacing solvent molecule 1545 (atom 8390) with NA
  >> Replacing solvent molecule 2157 (atom 10226) with NA
  >> Replacing solvent molecule 1717 (atom 8906) with NA
  >> Replacing solvent molecule 1404 (atom 7967) with NA
  >> Replacing solvent molecule 1176 (atom 7283) with NA
  >> Replacing solvent molecule 2032 (atom 9851) with NA
  >> Replacing solvent molecule 852 (atom 6311) with NA
  >> Replacing solvent molecule 1266 (atom 7553) with NA
  >> Replacing solvent molecule 829 (atom 6242) with NA
  >> Replacing solvent molecule 1770 (atom 9065) with NA
  >> Replacing solvent molecule 1939 (atom 9572) with NA
  >> Replacing solvent molecule 2145 (atom 10190) with NA
  >> Replacing solvent molecule 2200 (atom 10355) with C

In [ ]:
# ── CELL 5: Energy Minimisation ───────────────────────────────────────────────

os.chdir(WORK_DIR)

# ── Step 5a: grompp — prepare EM run input ────────────────────────────────────
print("Step 5a: grompp for energy minimisation...")
result = subprocess.run(
    'gmx grompp -f em.mdp -c ionised.gro -p topol_complex.top -o em.tpr -maxwarn 15',
    shell=True, capture_output=True, text=True
)

if result.returncode != 0:
    print(result.stdout[-500:])
    print(result.stderr[-1500:])
    raise RuntimeError("grompp for EM failed.")

assert (WORK_DIR / 'em.tpr').exists()
print("  em.tpr created successfully")

# ── Step 5b: mdrun — run energy minimisation ──────────────────────────────────
# -v  : verbose (prints progress every step)
# -nt : number of threads (0 = auto-detect)
print("\nStep 5b: Running energy minimisation (5-10 minutes)...")
result = subprocess.run(
    'gmx mdrun -v -deffnm em -nt 0',
    shell=True, capture_output=True, text=True
)

print(result.stdout[-1000:])
if result.stderr:
    print(result.stderr[-1000:])

if result.returncode != 0:
    raise RuntimeError("Energy minimisation failed.")

# ── Verify outputs ────────────────────────────────────────────────────────────
em_gro = WORK_DIR / 'em.gro'
em_edr = WORK_DIR / 'em.edr'
assert em_gro.exists(), "ERROR: em.gro not created"
assert em_edr.exists(), "ERROR: em.edr not created"
print(f"\n  em.gro created ({em_gro.stat().st_size / 1024:.1f} KB)")
print(f"  em.edr created ({em_edr.stat().st_size / 1024:.1f} KB)")

# ── Extract final potential energy ────────────────────────────────────────────
print("\nExtracting final potential energy...")
result2 = subprocess.run(
    'echo "Potential" | gmx energy -f em.edr -o em_energy.xvg',
    shell=True, capture_output=True, text=True
)
for line in (result2.stdout + result2.stderr).splitlines():
    if 'Potential' in line and ('kJ' in line or 'Total' in line or '-' in line):
        print(f"  >> {line.strip()}")

print("\n" + "=" * 60)
print("  CELL 5 COMPLETE")
print("  Energy minimisation done")
print("  Next: Cell 6 — NVT equilibration (heat to 310 K)")
print("=" * 60)

Step 5a: grompp for energy minimisation...
  em.tpr created successfully

Step 5b: Running energy minimisation (5-10 minutes)...

om= 3372
Step=  462, Dmax= 1.0e-02 nm, Epot= -1.79030e+05 Fmax= 4.08630e+03, atom= 3372
Step=  463, Dmax= 1.2e-02 nm, Epot= -1.78940e+05 Fmax= 1.43684e+04, atom= 3372
Step=  464, Dmax= 6.0e-03 nm, Epot= -1.79055e+05 Fmax= 5.14186e+03, atom= 3372
Step=  465, Dmax= 7.2e-03 nm, Epot= -1.79080e+05 Fmax= 5.93128e+03, atom= 3372
Step=  466, Dmax= 8.7e-03 nm, Epot= -1.79097e+05 Fmax= 7.35529e+03, atom= 3372
Step=  467, Dmax= 1.0e-02 nm, Epot= -1.79112e+05 Fmax= 8.58314e+03, atom= 3372
Step=  468, Dmax= 1.3e-02 nm, Epot= -1.79114e+05 Fmax= 1.05384e+04, atom= 3372
Step=  469, Dmax= 1.5e-02 nm, Epot= -1.79110e+05 Fmax= 1.23915e+04, atom= 3372
Step=  470, Dmax= 7.5e-03 nm, Epot= -1.79220e+05 Fmax= 9.41509e+02, atom= 3372

writing lowest energy coordinates.

Steepest Descents converged to Fmax < 1000 in 471 steps
Potential Energy  = -1.7922041e+05
Maximum force     =  9

In [ ]:
# ── CELL 0b: Session Recovery — Restore paths and environment ─────────────────

import os
import subprocess
from pathlib import Path

# Restore Drive paths
DRIVE_BASE  = Path('/content/drive/MyDrive/PfDHFR_MD')
SYSTEMS_DIR = DRIVE_BASE / 'systems'
RESULTS_DIR = DRIVE_BASE / 'md_results'

# Restore work directory
WORK_DIR = Path('/content/md_work/pyrimethamine')

# Restore GROMACS to PATH
os.environ['PATH'] = '/content/miniforge/bin:' + os.environ['PATH']

# Verify everything still exists locally
print("Checking local files...")
for fname in ['em.gro', 'em.edr', 'topol_complex.top', 'ionised.gro',
              'pyrimethamine_GMX.itp', 'posre.itp',
              'nvt.mdp', 'npt.mdp', 'md.mdp']:
    f = WORK_DIR / fname
    status = "OK" if f.exists() else "MISSING"
    print(f"  {fname:35s} : {status}")

# Verify GROMACS
result = subprocess.run('gmx --version', shell=True, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if 'GROMACS version' in line:
        print(f"\n  {line.strip()}")
        break

print("\n  Recovery complete — ready for Cell 6 (NVT equilibration)")

Checking local files...
  em.gro                              : OK
  em.edr                              : OK
  topol_complex.top                   : OK
  ionised.gro                         : OK
  pyrimethamine_GMX.itp               : OK
  posre.itp                           : OK
  nvt.mdp                             : OK
  npt.mdp                             : OK
  md.mdp                              : OK

  GROMACS version:     2026.0-conda_forge

  Recovery complete — ready for Cell 6 (NVT equilibration)


In [ ]:
# ── CELL 6 FIX 5: Run NVT on CPU (no GPU flags) ──────────────────────────────

os.chdir(WORK_DIR)

print("Running NVT equilibration on CPU (15-25 minutes)...")
result = subprocess.run(
    'gmx mdrun -deffnm nvt -nt 0',
    shell=True, capture_output=True, text=True,
    timeout=3600
)

print(result.stdout[-1000:])
if result.stderr:
    print(result.stderr[-500:])

if result.returncode != 0:
    raise RuntimeError("NVT equilibration failed.")

nvt_gro = WORK_DIR / 'nvt.gro'
nvt_edr = WORK_DIR / 'nvt.edr'
assert nvt_gro.exists(), "ERROR: nvt.gro not created"
assert nvt_edr.exists(), "ERROR: nvt.edr not created"
print(f"\n  nvt.gro created ({nvt_gro.stat().st_size / 1024:.1f} KB)")
print(f"  nvt.edr created ({nvt_edr.stat().st_size / 1024:.1f} KB)")

# ── Extract final temperature ─────────────────────────────────────────────────
print("\nExtracting final temperature...")
result2 = subprocess.run(
    'echo "Temperature" | gmx energy -f nvt.edr -o nvt_temp.xvg',
    shell=True, capture_output=True, text=True
)
for line in (result2.stdout + result2.stderr).splitlines():
    if 'Temperature' in line and ('Average' in line or 'Std' in line):
        print(f"  >> {line.strip()}")

print("\n" + "=" * 60)
print("  CELL 6 COMPLETE")
print("  NVT equilibration done — system heated to 310 K")
print("  Next: Cell 7 — NPT equilibration (pressurise to 1 bar)")
print("=" * 60)

Running NVT equilibration on CPU (15-25 minutes)...

penMP threads 

starting mdrun 'Protein'
50000 steps,    100.0 ps.

Writing final coordinates.

               Core t (s)   Wall t (s)        (%)
       Time:     1394.545      697.273      200.0
                 (ns/day)    (hour/ns)    (ms/step)  (Matom*steps/s) 
Performance:       12.391        1.937       13.945            0.966 

GROMACS reminds you: "During my undergraduate work I concluded that electrostatics is unlikely to be important [for enzymes]" (Arieh Warshel, Nobel lecture 2013)



  nvt.gro created (908.2 KB)
  nvt.edr created (55.1 KB)

Extracting final temperature...

  CELL 6 COMPLETE
  NVT equilibration done — system heated to 310 K
  Next: Cell 7 — NPT equilibration (pressurise to 1 bar)


In [ ]:
# ── CELL 7: NPT Equilibration — Pressurise to 1 bar ──────────────────────────

os.chdir(WORK_DIR)

# ── Step 7a: grompp for NPT ───────────────────────────────────────────────────
print("Step 7a: grompp for NPT equilibration...")
result = subprocess.run(
    'gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt '
    '-p topol_complex.top -n index.ndx -o npt.tpr -maxwarn 15',
    shell=True, capture_output=True, text=True
)

if result.returncode != 0:
    print(result.stdout[-500:])
    print(result.stderr[-2000:])
    raise RuntimeError("grompp for NPT failed.")

assert (WORK_DIR / 'npt.tpr').exists()
print("  npt.tpr created successfully")

# ── Step 7b: mdrun NPT ────────────────────────────────────────────────────────
print("\nStep 7b: Running NPT equilibration (15-25 minutes)...")
result = subprocess.run(
    'gmx mdrun -deffnm npt -nt 0',
    shell=True, capture_output=True, text=True,
    timeout=3600
)

print(result.stdout[-1000:])
if result.stderr:
    print(result.stderr[-500:])

if result.returncode != 0:
    raise RuntimeError("NPT equilibration failed.")

npt_gro = WORK_DIR / 'npt.gro'
npt_edr = WORK_DIR / 'npt.edr'
assert npt_gro.exists(), "ERROR: npt.gro not created"
assert npt_edr.exists(), "ERROR: npt.edr not created"
print(f"\n  npt.gro created ({npt_gro.stat().st_size / 1024:.1f} KB)")
print(f"  npt.edr created ({npt_edr.stat().st_size / 1024:.1f} KB)")

# ── Extract final pressure and density ───────────────────────────────────────
print("\nExtracting pressure and density...")
result2 = subprocess.run(
    'printf "Pressure\nDensity\n" | gmx energy -f npt.edr -o npt_energy.xvg',
    shell=True, capture_output=True, text=True
)
for line in (result2.stdout + result2.stderr).splitlines():
    if ('Pressure' in line or 'Density' in line) and ('Average' in line or 'Std' in line):
        print(f"  >> {line.strip()}")

print("\n" + "=" * 60)
print("  CELL 7 COMPLETE")
print("  NPT equilibration done — system at 310 K, 1 bar")
print("  Next: Cell 8 — production MD (50 ns in 5 x 10 ns chunks)")
print("=" * 60)

Step 7a: grompp for NPT equilibration...
  npt.tpr created successfully

Step 7b: Running NPT equilibration (15-25 minutes)...

thread
Using 2 OpenMP threads 

starting mdrun 'Protein'
50000 steps,    100.0 ps.

Writing final coordinates.

               Core t (s)   Wall t (s)        (%)
       Time:     1489.214      744.607      200.0
                 (ns/day)    (hour/ns)    (ms/step)  (Matom*steps/s) 
Performance:       11.604        2.068       14.892            0.905 

GROMACS reminds you: "Computers are incredibly fast, accurate and stupid. Humans are incredibly slow, inaccurate and... also stupid." (Anonymous)



  npt.gro created (908.2 KB)
  npt.edr created (67.1 KB)

Extracting pressure and density...

  CELL 7 COMPLETE
  NPT equilibration done — system at 310 K, 1 bar
  Next: Cell 8 — production MD (50 ns in 5 x 10 ns chunks)


In [ ]:
# ── CELL 7e: Compile GROMACS with CUDA support ───────────────────────────────

import subprocess, os

print("Compiling GROMACS with CUDA support (~45 minutes)...")
print("Do not close this tab.\n")

cmds = [
    # Dependencies
    'apt-get install -y -q cmake libfftw3-dev',
    # Download GROMACS 2023.3 source (stable, known CUDA compatibility)
    'wget -q https://ftp.gromacs.org/gromacs/gromacs-2023.3.tar.gz -O /content/gromacs.tar.gz',
    # Extract
    'tar -xzf /content/gromacs.tar.gz -C /content/',
    # Configure with CUDA
    ('cmake /content/gromacs-2023.3 '
     '-B /content/gromacs_build '
     '-DGMX_BUILD_OWN_FFTW=OFF '
     '-DREGRESSIONTEST_DOWNLOAD=OFF '
     '-DGMX_GPU=CUDA '
     '-DCUDA_TOOLKIT_ROOT_DIR=/usr/local/cuda '
     '-DGMX_SIMD=AVX2_256 '
     '-DCMAKE_INSTALL_PREFIX=/content/gromacs_cuda '
     '-DGMX_BUILD_MDRUN_ONLY=ON '
     '-DBUILD_SHARED_LIBS=OFF '
     '-DGMX_OPENMP=ON '
     '-DGMX_MPI=OFF'),
    # Build (use all available cores)
    'cmake --build /content/gromacs_build -j$(nproc) --target install',
]

for i, cmd in enumerate(cmds, 1):
    print(f"Step {i}/{len(cmds)}: {cmd[:70]}...")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ERROR:")
        print(result.stderr[-1000:])
        raise RuntimeError(f"Step {i} failed.")
    print(f"  Done.")

# Point to CUDA GROMACS
os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']

# Verify
result = subprocess.run('gmx_mpi --version 2>/dev/null || gmx --version',
                        shell=True, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if 'GROMACS' in line or 'CUDA' in line or 'GPU' in line:
        print(f"  {line.strip()}")

print("\n  CUDA GROMACS ready.")

Compiling GROMACS with CUDA support (~45 minutes)...
Do not close this tab.

Step 1/5: apt-get install -y -q cmake libfftw3-dev...
  Done.
Step 2/5: wget -q https://ftp.gromacs.org/gromacs/gromacs-2023.3.tar.gz -O /cont...
  Done.
Step 3/5: tar -xzf /content/gromacs.tar.gz -C /content/...
  Done.
Step 4/5: cmake /content/gromacs-2023.3 -B /content/gromacs_build -DGMX_BUILD_OW...
  Done.
Step 5/5: cmake --build /content/gromacs_build -j$(nproc) --target install...
  Done.
  :-) GROMACS - gmx, 2023.3 (-:
  GROMACS version:    2023.3
  GPU support:        CUDA
  GPU FFT library:    cuFFT
  Multi-GPU FFT:      none
  CUDA compiler:      /usr/local/cuda/bin/nvcc nvcc: NVIDIA (R) Cuda compiler driver;Copyright (c) 2005-2025 NVIDIA Corporation;Built on Fri_Feb_21_20:23:50_PST_2025;Cuda compilation tools, release 12.8, V12.8.93;Build cuda_12.8.r12.8/compiler.35583870_0
  CUDA compiler flags:-std=c++17;--generate-code=arch=compute_50,code=sm_50;--generate-code=arch=compute_52,code=sm_52;--gener

In [ ]:
# ── CELL 7f: Verify GPU detection with CUDA GROMACS ──────────────────────────

import subprocess, os
from pathlib import Path

WORK_DIR = Path('/content/md_work/pyrimethamine')
os.chdir(WORK_DIR)

# Ensure CUDA GROMACS is first in PATH
os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']

# Quick 100-step test with GPU
print("Testing GPU mdrun with CUDA GROMACS...")
result = subprocess.run(
    'gmx mdrun -s npt.tpr -o test.trr -e test.edr -g test.log '
    '-c test.gro -nb gpu -gpu_id 0 -nsteps 100 -nt 0',
    shell=True, capture_output=True, text=True, timeout=120
)

print(result.stdout[-500:])
if result.stderr:
    print(result.stderr[-500:])

if result.returncode == 0:
    print("\n  GPU test: PASSED — T4 is working with CUDA GROMACS")
    print("  Ready for Cell 8 — production MD at full GPU speed")
else:
    print("\n  GPU test failed — check output above")

Testing GPU mdrun with CUDA GROMACS...

_512 (see log).
Reading file npt.tpr, VERSION 2026.0-conda_forge (single precision)

-------------------------------------------------------
Program:     gmx mdrun, version 2023.3
Source file: src/gromacs/fileio/tpxio.cpp (line 2833)

Fatal error:
reading tpx file (npt.tpr) version 138 with version 129 program

For more information and tips for troubleshooting, please check the GROMACS
website at http://www.gromacs.org/Documentation/Errors
-------------------------------------------------------


  GPU test failed — check output above


In [ ]:
# ── CELL 7g: Regenerate TPR files with CUDA GROMACS 2023.3 ───────────────────

import subprocess, os
from pathlib import Path

WORK_DIR = Path('/content/md_work/pyrimethamine')
os.chdir(WORK_DIR)

# Ensure CUDA GROMACS is first in PATH
os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']

# Verify which gmx is being used
result = subprocess.run('which gmx && gmx --version | grep "GROMACS version"',
                        shell=True, capture_output=True, text=True)
print(result.stdout.strip())

# Regenerate npt.tpr with GROMACS 2023.3
print("\nRegenerating npt.tpr with GROMACS 2023.3...")
result = subprocess.run(
    'gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt '
    '-p topol_complex.top -n index.ndx -o npt2.tpr -maxwarn 15',
    shell=True, capture_output=True, text=True
)

if result.returncode != 0:
    print(result.stdout[-500:])
    print(result.stderr[-1500:])
    raise RuntimeError("grompp failed.")

print("  npt2.tpr created")

# Quick GPU test with new TPR
print("\nTesting GPU with npt2.tpr...")
result = subprocess.run(
    'gmx mdrun -s npt2.tpr -o test.trr -e test.edr -g test.log '
    '-c test.gro -nb gpu -gpu_id 0 -nsteps 100 -nt 0',
    shell=True, capture_output=True, text=True, timeout=120
)

print(result.stdout[-500:])
if result.stderr:
    print(result.stderr[-300:])

if result.returncode == 0:
    print("\n  GPU test: PASSED")
else:
    print("\n  GPU test failed")

/content/gromacs_cuda/bin/gmx
GROMACS version:    2023.3

Regenerating npt.tpr with GROMACS 2023.3...
  npt2.tpr created

Testing GPU with npt2.tpr...

Protein'
100 steps,      0.2 ps.

Writing final coordinates.

               Core t (s)   Wall t (s)        (%)
       Time:        0.591        0.298      198.5
                 (ns/day)    (hour/ns)
Performance:       58.659        0.409

GROMACS reminds you: "Let's Go Hang Out In a Mall" (LIVE)



  GPU test: PASSED


In [ ]:
# ── CELL 8: Production MD — 50 ns in 5 x 10 ns chunks ───────────────────────

import subprocess, os, shutil
from pathlib import Path

WORK_DIR = Path('/content/md_work/pyrimethamine')
DRIVE_BASE = Path('/content/drive/MyDrive/PfDHFR_MD')
RESULTS_DIR = DRIVE_BASE / 'md_results' / 'pyrimethamine'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']

CHUNK_NS   = 10        # ns per chunk
CHUNK_STEPS = 5_000_000  # 10 ns at 2 fs timestep
N_CHUNKS   = 5

print("=" * 60)
print("  Production MD — pyrimethamine reference")
print(f"  Total: {CHUNK_NS * N_CHUNKS} ns in {N_CHUNKS} x {CHUNK_NS} ns chunks")
print(f"  GPU: T4 | Expected speed: ~58 ns/day")
print(f"  Expected time per chunk: ~4 hours")
print("=" * 60)

for chunk in range(1, N_CHUNKS + 1):
    print(f"\n{'─'*60}")
    print(f"  CHUNK {chunk}/{N_CHUNKS}  ({(chunk-1)*CHUNK_NS}–{chunk*CHUNK_NS} ns)")
    print(f"{'─'*60}")

    out_prefix = f'md_chunk{chunk}'

    # ── Prepare TPR for this chunk ────────────────────────────────────────────
    if chunk == 1:
        # First chunk starts from NPT equilibrated structure
        grompp_cmd = (
            f'gmx grompp -f md.mdp -c npt.gro -t npt.cpt '
            f'-p topol_complex.top -n index.ndx '
            f'-o {out_prefix}.tpr -maxwarn 15'
        )
    else:
        # Subsequent chunks extend from previous checkpoint
        prev_cpt = f'md_chunk{chunk-1}.cpt'
        prev_tpr = f'md_chunk{chunk-1}.tpr'
        grompp_cmd = (
            f'gmx grompp -f md.mdp -c md_chunk{chunk-1}.gro '
            f'-t {prev_cpt} -p topol_complex.top -n index.ndx '
            f'-o {out_prefix}.tpr -maxwarn 15'
        )

    print(f"  grompp for chunk {chunk}...")
    result = subprocess.run(grompp_cmd, shell=True,
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-300:])
        print(result.stderr[-1000:])
        raise RuntimeError(f"grompp chunk {chunk} failed.")
    print(f"  {out_prefix}.tpr created")

    # ── Run mdrun for this chunk ──────────────────────────────────────────────
    print(f"  Running mdrun chunk {chunk} (~4 hours)...")
    mdrun_cmd = (
        f'gmx mdrun -deffnm {out_prefix} '
        f'-nb gpu -gpu_id 0 -nt 0 '
        f'-nsteps {CHUNK_STEPS}'
    )
    result = subprocess.run(mdrun_cmd, shell=True,
                            capture_output=True, text=True,
                            timeout=18000)  # 5 hour safety timeout

    # Print performance summary
    for line in (result.stdout + result.stderr).splitlines():
        if 'Performance' in line or 'ns/day' in line or 'steps' in line:
            print(f"  >> {line.strip()}")

    if result.returncode != 0:
        print(result.stderr[-1000:])
        raise RuntimeError(f"mdrun chunk {chunk} failed.")

    # ── Save chunk outputs to Drive immediately ───────────────────────────────
    print(f"  Saving chunk {chunk} to Drive...")
    for ext in ['.xtc', '.gro', '.cpt', '.edr', '.log']:
        src = WORK_DIR / f'{out_prefix}{ext}'
        if src.exists():
            shutil.copy2(src, RESULTS_DIR / src.name)
            print(f"    {src.name} → Drive ({src.stat().st_size / 1024 / 1024:.1f} MB)")

    print(f"  Chunk {chunk} complete and saved to Drive.")

print("\n" + "=" * 60)
print("  CELL 8 COMPLETE")
print(f"  50 ns production MD finished for pyrimethamine")
print(f"  All trajectories saved to: {RESULTS_DIR}")
print("  Next: Cell 9 — verify outputs and wrap up session")
print("=" * 60)

  Production MD — pyrimethamine reference
  Total: 50 ns in 5 x 10 ns chunks
  GPU: T4 | Expected speed: ~58 ns/day
  Expected time per chunk: ~4 hours

────────────────────────────────────────────────────────────
  CHUNK 1/5  (0–10 ns)
────────────────────────────────────────────────────────────
  grompp for chunk 1...
  md_chunk1.tpr created
  Running mdrun chunk 1 (~4 hours)...
  >> gmx mdrun -deffnm md_chunk1 -nb gpu -gpu_id 0 -nt 0 -nsteps 5000000
  >> Overriding nsteps with value passed on the command line: 5000000 steps, 1e+04 ps
  >> 5000000 steps,  10000.0 ps.
  >> (ns/day)    (hour/ns)
  >> Performance:      218.573        0.110
  Saving chunk 1 to Drive...
    md_chunk1.xtc → Drive (46.8 MB)
    md_chunk1.gro → Drive (0.9 MB)
    md_chunk1.cpt → Drive (0.3 MB)
    md_chunk1.edr → Drive (0.7 MB)
    md_chunk1.log → Drive (0.6 MB)
  Chunk 1 complete and saved to Drive.

────────────────────────────────────────────────────────────
  CHUNK 2/5  (10–20 ns)
───────────────────────

In [ ]:
# ── CELL 8 RECOVERY: Resume production MD from last completed chunk ───────────

import subprocess, os, shutil
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

WORK_DIR    = Path('/content/md_work/pyrimethamine')
DRIVE_BASE  = Path('/content/drive/MyDrive/PfDHFR_MD')
RESULTS_DIR = DRIVE_BASE / 'md_results' / 'pyrimethamine'

os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']

# ── Check Drive for completed chunks ─────────────────────────────────────────
print("Checking Drive for completed chunks...")
completed_chunks = []
for chunk in range(1, 6):
    xtc = RESULTS_DIR / f'md_chunk{chunk}.xtc'
    gro = RESULTS_DIR / f'md_chunk{chunk}.gro'
    cpt = RESULTS_DIR / f'md_chunk{chunk}.cpt'
    if xtc.exists() and gro.exists() and cpt.exists():
        print(f"  Chunk {chunk}: COMPLETE ({xtc.stat().st_size/1024/1024:.1f} MB)")
        completed_chunks.append(chunk)
    else:
        print(f"  Chunk {chunk}: missing")

# ── Check local files ─────────────────────────────────────────────────────────
print("\nChecking local work directory...")
work_exists = WORK_DIR.exists()
print(f"  Work dir exists: {work_exists}")
if work_exists:
    for f in sorted(WORK_DIR.glob('md_chunk*')):
        print(f"  {f.name}: {f.stat().st_size/1024/1024:.1f} MB")

# ── Check GROMACS ─────────────────────────────────────────────────────────────
print("\nChecking GROMACS...")
result = subprocess.run('gmx --version 2>/dev/null || echo "not found"',
                        shell=True, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if 'GROMACS version' in line or 'not found' in line:
        print(f"  {line.strip()}")

if completed_chunks:
    print(f"\n  Last completed chunk: {max(completed_chunks)}")
    print(f"  Resume from chunk: {max(completed_chunks) + 1}")

Mounted at /content/drive
Checking Drive for completed chunks...
  Chunk 1: COMPLETE (46.8 MB)
  Chunk 2: COMPLETE (46.8 MB)
  Chunk 3: missing
  Chunk 4: missing
  Chunk 5: missing

Checking local work directory...
  Work dir exists: False

Checking GROMACS...
  not found

  Last completed chunk: 2
  Resume from chunk: 3


In [ ]:
# ── CELL 8 FULL RECOVERY: Rebuild environment and resume from chunk 3 ─────────

import subprocess, os, shutil, zipfile
from pathlib import Path

DRIVE_BASE  = Path('/content/drive/MyDrive/PfDHFR_MD')
SYSTEMS_DIR = DRIVE_BASE / 'systems'
RESULTS_DIR = DRIVE_BASE / 'md_results' / 'pyrimethamine'
WORK_DIR    = Path('/content/md_work/pyrimethamine')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

# ── Step 1: Install miniforge + GROMACS 2026 (for solvate/grompp prep) ───────
print("Step 1: Installing miniforge + GROMACS...")
os.environ['PATH'] = '/content/miniforge/bin:' + os.environ['PATH']
if not Path('/content/miniforge/bin/conda').exists():
    subprocess.run('wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O miniforge.sh', shell=True)
    subprocess.run('bash miniforge.sh -b -p /content/miniforge', shell=True)
    subprocess.run('/content/miniforge/bin/conda install -y -c conda-forge gromacs -q', shell=True, capture_output=True)
    print("  Miniforge + GROMACS installed")
else:
    print("  Miniforge already present")

# ── Step 2: Compile CUDA GROMACS 2023.3 ──────────────────────────────────────
print("\nStep 2: Compiling CUDA GROMACS 2023.3 (~45 minutes)...")
os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']
if not Path('/content/gromacs_cuda/bin/gmx').exists():
    cmds = [
        'apt-get install -y -q cmake libfftw3-dev',
        'wget -q https://ftp.gromacs.org/gromacs/gromacs-2023.3.tar.gz -O /content/gromacs.tar.gz',
        'tar -xzf /content/gromacs.tar.gz -C /content/',
        ('cmake /content/gromacs-2023.3 -B /content/gromacs_build '
         '-DGMX_BUILD_OWN_FFTW=OFF -DREGRESSIONTEST_DOWNLOAD=OFF '
         '-DGMX_GPU=CUDA -DCUDA_TOOLKIT_ROOT_DIR=/usr/local/cuda '
         '-DGMX_SIMD=AVX2_256 -DCMAKE_INSTALL_PREFIX=/content/gromacs_cuda '
         '-DGMX_BUILD_MDRUN_ONLY=ON -DBUILD_SHARED_LIBS=OFF '
         '-DGMX_OPENMP=ON -DGMX_MPI=OFF'),
        'cmake --build /content/gromacs_build -j$(nproc) --target install',
    ]
    for i, cmd in enumerate(cmds, 1):
        print(f"  [{i}/5] {cmd[:60]}...")
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if result.returncode != 0:
            print(result.stderr[-500:])
            raise RuntimeError(f"CUDA GROMACS build step {i} failed.")
    print("  CUDA GROMACS compiled successfully")
else:
    print("  CUDA GROMACS already compiled")

# Verify
result = subprocess.run('gmx --version', shell=True, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if 'GROMACS version' in line or 'GPU support' in line:
        print(f"  {line.strip()}")

# ── Step 3: Unzip system files ────────────────────────────────────────────────
print("\nStep 3: Extracting system files...")
with zipfile.ZipFile(SYSTEMS_DIR / 'pyrimethamine_system.zip', 'r') as zf:
    zf.extractall(WORK_DIR)
print("  Extracted.")

# ── Step 4: Patch topology paths ──────────────────────────────────────────────
print("\nStep 4: Patching topology paths...")
import re
top_file = WORK_DIR / 'topol_complex.top'
top_text = top_file.read_text(encoding='utf-8')
top_fixed = top_text.replace(
    '../ligand_params/pyrimethamine/pyrimethamine_GMX.itp',
    'pyrimethamine_GMX.itp'
)
top_fixed = re.sub(r'#include\s+"[^"]*posre\.itp"', '#include "posre.itp"', top_fixed)
top_fixed = top_fixed.replace('LIG                  1', 'pyrimethamine        1')
top_file.write_text(top_fixed, encoding='utf-8')
print("  Topology patched.")

# ── Step 5: Restore chunk 2 outputs from Drive (needed for continuation) ──────
print("\nStep 5: Restoring chunk 2 checkpoint from Drive...")
for ext in ['.gro', '.cpt', '.tpr']:
    src = RESULTS_DIR / f'md_chunk2{ext}'
    if src.exists():
        shutil.copy2(src, WORK_DIR / f'md_chunk2{ext}')
        print(f"  md_chunk2{ext} restored")
    else:
        print(f"  WARNING: md_chunk2{ext} not found on Drive")

# ── Step 6: Rebuild solvated/ionised system for index file generation ─────────
print("\nStep 6: Rebuilding index file...")
os.environ['PATH'] = '/content/miniforge/bin:' + os.environ['PATH']

# Solvate
subprocess.run('gmx solvate -cp complex.gro -cs spc216.gro -o solvated.gro -p topol_complex.top',
               shell=True, capture_output=True)
# Ions grompp
subprocess.run('gmx grompp -f em.mdp -c solvated.gro -p topol_complex.top -o ions.tpr -maxwarn 15',
               shell=True, capture_output=True)
# Genion
subprocess.run('echo "SOL" | gmx genion -s ions.tpr -o ionised.gro -p topol_complex.top '
               '-pname NA -nname CL -neutral -conc 0.15',
               shell=True, capture_output=True)
# Make index
ndx_commands = "1 | 13\nname 21 Protein_LIG\nq\n"
subprocess.run(['gmx', 'make_ndx', '-f', 'ionised.gro', '-o', 'index.ndx'],
               input=ndx_commands, capture_output=True, text=True)

# Verify index
ndx_text = (WORK_DIR / 'index.ndx').read_text()
if 'Protein_LIG' not in ndx_text:
    ndx_fixed = re.sub(r'\[ Protein_UNL \]', '[ Protein_LIG ]', ndx_text)
    (WORK_DIR / 'index.ndx').write_text(ndx_fixed)
print(f"  index.ndx ready ({'Protein_LIG' in (WORK_DIR / 'index.ndx').read_text()})")

# ── Step 7: Resume production MD from chunk 3 ─────────────────────────────────
print("\nStep 7: Resuming production MD from chunk 3...")
os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']

CHUNK_STEPS = 5_000_000
N_CHUNKS    = 5

for chunk in range(3, N_CHUNKS + 1):
    print(f"\n{'─'*60}")
    print(f"  CHUNK {chunk}/{N_CHUNKS}  ({(chunk-1)*10}–{chunk*10} ns)")
    print(f"{'─'*60}")

    out_prefix = f'md_chunk{chunk}'
    prev = f'md_chunk{chunk-1}'

    grompp_cmd = (
        f'gmx grompp -f md.mdp -c {prev}.gro -t {prev}.cpt '
        f'-p topol_complex.top -n index.ndx '
        f'-o {out_prefix}.tpr -maxwarn 15'
    )
    print(f"  grompp for chunk {chunk}...")
    result = subprocess.run(grompp_cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-300:])
        print(result.stderr[-1000:])
        raise RuntimeError(f"grompp chunk {chunk} failed.")
    print(f"  {out_prefix}.tpr created")

    print(f"  Running mdrun chunk {chunk} (~1 hour at 218 ns/day)...")
    mdrun_cmd = (
        f'gmx mdrun -deffnm {out_prefix} '
        f'-nb gpu -gpu_id 0 -nt 0 -nsteps {CHUNK_STEPS}'
    )
    result = subprocess.run(mdrun_cmd, shell=True, capture_output=True, text=True,
                            timeout=18000)

    for line in (result.stdout + result.stderr).splitlines():
        if 'Performance' in line or 'steps,' in line:
            print(f"  >> {line.strip()}")

    if result.returncode != 0:
        print(result.stderr[-1000:])
        raise RuntimeError(f"mdrun chunk {chunk} failed.")

    print(f"  Saving chunk {chunk} to Drive...")
    for ext in ['.xtc', '.gro', '.cpt', '.edr', '.log']:
        src = WORK_DIR / f'{out_prefix}{ext}'
        if src.exists():
            shutil.copy2(src, RESULTS_DIR / src.name)
            print(f"    {src.name} → Drive ({src.stat().st_size/1024/1024:.1f} MB)")
    print(f"  Chunk {chunk} complete.")

print("\n" + "=" * 60)
print("  RECOVERY COMPLETE")
print("  50 ns production MD finished for pyrimethamine")
print(f"  All trajectories on Drive: {RESULTS_DIR}")
print("=" * 60)

Step 1: Installing miniforge + GROMACS...
  Miniforge + GROMACS installed

Step 2: Compiling CUDA GROMACS 2023.3 (~45 minutes)...
  [1/5] apt-get install -y -q cmake libfftw3-dev...
  [2/5] wget -q https://ftp.gromacs.org/gromacs/gromacs-2023.3.tar.g...
  [3/5] tar -xzf /content/gromacs.tar.gz -C /content/...
  [4/5] cmake /content/gromacs-2023.3 -B /content/gromacs_build -DGM...
  [5/5] cmake --build /content/gromacs_build -j$(nproc) --target ins...
  CUDA GROMACS compiled successfully
  GROMACS version:    2023.3
  GPU support:        CUDA

Step 3: Extracting system files...
  Extracted.

Step 4: Patching topology paths...
  Topology patched.

Step 5: Restoring chunk 2 checkpoint from Drive...
  md_chunk2.gro restored
  md_chunk2.cpt restored

Step 6: Rebuilding index file...
  index.ndx ready (True)

Step 7: Resuming production MD from chunk 3...

────────────────────────────────────────────────────────────
  CHUNK 3/5  (20–30 ns)
────────────────────────────────────────────────────

In [ ]:
# ── FIX: Install FFTW3, then rebuild index and run chunks 4 & 5 ──────────────

import subprocess, os, re, shutil
from pathlib import Path

DRIVE_BASE  = Path('/content/drive/MyDrive/PfDHFR_MD')
RESULTS_DIR = DRIVE_BASE / 'md_results' / 'pyrimethamine'
WORK_DIR    = Path('/content/md_work/pyrimethamine')
os.chdir(WORK_DIR)

# Install missing library
print("Installing libfftw3...")
subprocess.run('apt-get install -y -q libfftw3-dev', shell=True)
print("  Done.")

os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']

# Verify gmx works now
result = subprocess.run('gmx --version', shell=True, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if 'GROMACS version' in line or 'GPU support' in line:
        print(f"  {line.strip()}")

# Solvate
print("\nSolvating...")
r = subprocess.run('gmx solvate -cp complex.gro -cs spc216.gro -o solvated.gro -p topol_complex.top',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-500:])
    raise RuntimeError("solvate failed")
print("  Done.")

# grompp for genion
print("grompp for genion...")
r = subprocess.run('gmx grompp -f em.mdp -c solvated.gro -p topol_complex.top -o ions.tpr -maxwarn 15',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-500:])
    raise RuntimeError("grompp failed")
print("  Done.")

# Genion
print("Adding ions...")
r = subprocess.run('echo "SOL" | gmx genion -s ions.tpr -o ionised.gro '
                   '-p topol_complex.top -pname NA -nname CL -neutral -conc 0.15',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-500:])
    raise RuntimeError("genion failed")
print("  Done.")

# Make index
print("Building index...")
r = subprocess.run(['gmx', 'make_ndx', '-f', 'ionised.gro', '-o', 'index.ndx'],
                   input="1 | 13\nname 21 Protein_LIG\nq\n",
                   capture_output=True, text=True)

ndx_path = WORK_DIR / 'index.ndx'
assert ndx_path.exists(), f"index.ndx not created: {r.stderr[-300:]}"
ndx_text = ndx_path.read_text()
if 'Protein_LIG' not in ndx_text:
    ndx_text = re.sub(r'\[ Protein_UNL \]', '[ Protein_LIG ]', ndx_text)
    ndx_path.write_text(ndx_text)
print(f"  index.ndx ready (Protein_LIG: {'Protein_LIG' in ndx_path.read_text()})")

# Restore chunk 3 checkpoint
print("\nRestoring chunk 3 checkpoint...")
for ext in ['.gro', '.cpt']:
    src = RESULTS_DIR / f'md_chunk3{ext}'
    if src.exists():
        shutil.copy2(src, WORK_DIR / f'md_chunk3{ext}')
        print(f"  md_chunk3{ext} restored")

# Run chunks 4 and 5
print("\nRunning chunks 4 and 5...")
CHUNK_STEPS = 5_000_000

for chunk in range(4, 6):
    print(f"\n{'─'*60}")
    print(f"  CHUNK {chunk}/5  ({(chunk-1)*10}–{chunk*10} ns)")
    print(f"{'─'*60}")
    out_prefix = f'md_chunk{chunk}'
    prev = f'md_chunk{chunk-1}'

    r = subprocess.run(
        f'gmx grompp -f md.mdp -c {prev}.gro -t {prev}.cpt '
        f'-p topol_complex.top -n index.ndx -o {out_prefix}.tpr -maxwarn 15',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1000:])
        raise RuntimeError(f"grompp chunk {chunk} failed.")
    print(f"  {out_prefix}.tpr created")

    print(f"  Running mdrun chunk {chunk} (~1 hour)...")
    r = subprocess.run(
        f'gmx mdrun -deffnm {out_prefix} -nb gpu -gpu_id 0 -nt 0 -nsteps {CHUNK_STEPS}',
        shell=True, capture_output=True, text=True, timeout=18000)

    for line in (r.stdout + r.stderr).splitlines():
        if 'Performance' in line or 'steps,' in line:
            print(f"  >> {line.strip()}")

    if r.returncode != 0:
        print(r.stderr[-1000:])
        raise RuntimeError(f"mdrun chunk {chunk} failed.")

    print(f"  Saving chunk {chunk} to Drive...")
    for ext in ['.xtc', '.gro', '.cpt', '.edr', '.log']:
        src = WORK_DIR / f'{out_prefix}{ext}'
        if src.exists():
            shutil.copy2(src, RESULTS_DIR / src.name)
            print(f"    {src.name} → Drive ({src.stat().st_size/1024/1024:.1f} MB)")
    print(f"  Chunk {chunk} complete.")

print("\n" + "=" * 60)
print("  PYRIMETHAMINE 50 ns MD COMPLETE")
print("  All 5 chunks saved to Drive")
print("  Next: CNP0286261_0 (African NP, -11.86 kcal/mol)")
print("=" * 60)

Installing libfftw3...
  Done.
  GROMACS version:    2023.3
  GPU support:        CUDA

Solvating...
  Done.
grompp for genion...
  Done.
Adding ions...
  Done.
Building index...
  index.ndx ready (Protein_LIG: True)

Restoring chunk 3 checkpoint...
  md_chunk3.gro restored
  md_chunk3.cpt restored

Running chunks 4 and 5...

────────────────────────────────────────────────────────────
  CHUNK 4/5  (30–40 ns)
────────────────────────────────────────────────────────────
  md_chunk4.tpr created
  Running mdrun chunk 4 (~1 hour)...
  >> Overriding nsteps with value passed on the command line: 5000000 steps, 1e+04 ps
  >> 5000000 steps,  10000.0 ps.
  >> Performance:      221.176        0.109
  Saving chunk 4 to Drive...
    md_chunk4.xtc → Drive (46.8 MB)
    md_chunk4.gro → Drive (0.9 MB)
    md_chunk4.cpt → Drive (0.3 MB)
    md_chunk4.edr → Drive (0.7 MB)
    md_chunk4.log → Drive (0.6 MB)
  Chunk 4 complete.

────────────────────────────────────────────────────────────
  CHUNK 5/5  (4

In [ ]:
# ── FINAL RECOVERY: Pyrimethamine chunk 5 only ───────────────────────────────

import subprocess, os, re, shutil, zipfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE    = Path('/content/drive/MyDrive/PfDHFR_MD')
SYSTEMS_DIR   = DRIVE_BASE / 'systems'
RESULTS_DIR   = DRIVE_BASE / 'md_results' / 'pyrimethamine'
GROMACS_DRIVE = DRIVE_BASE / 'gromacs_cuda_binary'
WORK_DIR      = Path('/content/md_work/pyrimethamine')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

# ── Step 1: Install system dependencies ──────────────────────────────────────
print("Step 1: Installing dependencies...")
subprocess.run('apt-get install -y -q libfftw3-dev cmake', shell=True)
print("  Done.")

# ── Step 2: Restore GROMACS from Drive ───────────────────────────────────────
print("\nStep 2: Restoring CUDA GROMACS from Drive...")
os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']
if not Path('/content/gromacs_cuda/bin/gmx').exists():
    shutil.copytree(str(GROMACS_DRIVE), '/content/gromacs_cuda')
    subprocess.run('chmod -R +x /content/gromacs_cuda/bin/', shell=True)
    print("  Restored.")
else:
    print("  Already present.")

result = subprocess.run('gmx --version', shell=True, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if 'GROMACS version' in line or 'GPU support' in line:
        print(f"  {line.strip()}")

# ── Step 3: Extract and patch system files ────────────────────────────────────
print("\nStep 3: Extracting system files...")
with zipfile.ZipFile(SYSTEMS_DIR / 'pyrimethamine_system.zip', 'r') as zf:
    zf.extractall(WORK_DIR)
top_file = WORK_DIR / 'topol_complex.top'
top_text = top_file.read_text(encoding='utf-8')
top_fixed = top_text.replace('../ligand_params/pyrimethamine/pyrimethamine_GMX.itp', 'pyrimethamine_GMX.itp')
top_fixed = re.sub(r'#include\s+"[^"]*posre\.itp"', '#include "posre.itp"', top_fixed)
top_fixed = top_fixed.replace('LIG                  1', 'pyrimethamine        1')
top_file.write_text(top_fixed, encoding='utf-8')
print("  Done.")

# ── Step 4: Rebuild index file ────────────────────────────────────────────────
print("\nStep 4: Rebuilding index file...")
subprocess.run('gmx solvate -cp complex.gro -cs spc216.gro -o solvated.gro -p topol_complex.top',
               shell=True, capture_output=True)
subprocess.run('gmx grompp -f em.mdp -c solvated.gro -p topol_complex.top -o ions.tpr -maxwarn 15',
               shell=True, capture_output=True)
subprocess.run('echo "SOL" | gmx genion -s ions.tpr -o ionised.gro '
               '-p topol_complex.top -pname NA -nname CL -neutral -conc 0.15',
               shell=True, capture_output=True)
r = subprocess.run(['gmx', 'make_ndx', '-f', 'ionised.gro', '-o', 'index.ndx'],
                   input="1 | 13\nname 21 Protein_LIG\nq\n",
                   capture_output=True, text=True)
ndx_path = WORK_DIR / 'index.ndx'
assert ndx_path.exists(), f"index.ndx failed: {r.stderr[-200:]}"
ndx_text = ndx_path.read_text()
if 'Protein_LIG' not in ndx_text:
    ndx_path.write_text(re.sub(r'\[ Protein_UNL \]', '[ Protein_LIG ]', ndx_text))
print(f"  index.ndx ready (Protein_LIG: {'Protein_LIG' in ndx_path.read_text()})")

# ── Step 5: Restore chunk 4 checkpoint ───────────────────────────────────────
print("\nStep 5: Restoring chunk 4 checkpoint...")
for ext in ['.gro', '.cpt']:
    src = RESULTS_DIR / f'md_chunk4{ext}'
    assert src.exists(), f"md_chunk4{ext} not on Drive!"
    shutil.copy2(src, WORK_DIR / f'md_chunk4{ext}')
    print(f"  md_chunk4{ext} restored")

# ── Step 6: Run chunk 5 only ──────────────────────────────────────────────────
print("\n" + "─"*60)
print("  CHUNK 5/5  (40–50 ns)")
print("─"*60)

r = subprocess.run(
    'gmx grompp -f md.mdp -c md_chunk4.gro -t md_chunk4.cpt '
    '-p topol_complex.top -n index.ndx -o md_chunk5.tpr -maxwarn 15',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-1000:])
    raise RuntimeError("grompp chunk 5 failed.")
print("  md_chunk5.tpr created")

print("  Running mdrun chunk 5 (~1 hour)...")
r = subprocess.run(
    'gmx mdrun -deffnm md_chunk5 -nb gpu -gpu_id 0 -nt 0 -nsteps 5000000',
    shell=True, capture_output=True, text=True, timeout=18000)

for line in (r.stdout + r.stderr).splitlines():
    if 'Performance' in line or 'steps,' in line:
        print(f"  >> {line.strip()}")

if r.returncode != 0:
    print(r.stderr[-1000:])
    raise RuntimeError("mdrun chunk 5 failed.")

print("  Saving chunk 5 to Drive...")
for ext in ['.xtc', '.gro', '.cpt', '.edr', '.log']:
    src = WORK_DIR / f'md_chunk5{ext}'
    if src.exists():
        shutil.copy2(src, RESULTS_DIR / src.name)
        print(f"    {src.name} → Drive ({src.stat().st_size/1024/1024:.1f} MB)")

print("\n" + "=" * 60)
print("  PYRIMETHAMINE 50 ns MD COMPLETE")
print("  All 5 chunks (0–50 ns) saved to Drive")
print("  Next: CNP0286261_0 (African NP, -11.86 kcal/mol)")
print("=" * 60)

Mounted at /content/drive
Step 1: Installing dependencies...
  Done.

Step 2: Restoring CUDA GROMACS from Drive...
  Restored.
  GROMACS version:    2023.3
  GPU support:        CUDA

Step 3: Extracting system files...
  Done.

Step 4: Rebuilding index file...
  index.ndx ready (Protein_LIG: True)

Step 5: Restoring chunk 4 checkpoint...
  md_chunk4.gro restored
  md_chunk4.cpt restored

────────────────────────────────────────────────────────────
  CHUNK 5/5  (40–50 ns)
────────────────────────────────────────────────────────────
  md_chunk5.tpr created
  Running mdrun chunk 5 (~1 hour)...
  >> Overriding nsteps with value passed on the command line: 5000000 steps, 1e+04 ps
  >> 5000000 steps,  10000.0 ps.
  >> Performance:      228.450        0.105
  Saving chunk 5 to Drive...
    md_chunk5.xtc → Drive (46.8 MB)
    md_chunk5.gro → Drive (0.9 MB)
    md_chunk5.cpt → Drive (0.3 MB)
    md_chunk5.edr → Drive (0.7 MB)
    md_chunk5.log → Drive (0.6 MB)

  PYRIMETHAMINE 50 ns MD COMPLETE

In [ ]:
# ── CNP0286261_0: 100 ns MD — African NP Best Hit (-11.86 kcal/mol) ──────────

import subprocess, os, re, shutil, zipfile
from pathlib import Path

DRIVE_BASE    = Path('/content/drive/MyDrive/PfDHFR_MD')
SYSTEMS_DIR   = DRIVE_BASE / 'systems'
RESULTS_DIR   = DRIVE_BASE / 'md_results' / 'CNP0286261_0'
GROMACS_DRIVE = DRIVE_BASE / 'gromacs_cuda_binary'
WORK_DIR      = Path('/content/md_work/CNP0286261_0')
WORK_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']

# ── Step 1: Check zip exists ──────────────────────────────────────────────────
zip_file = SYSTEMS_DIR / 'CNP0286261_0_system.zip'
assert zip_file.exists(), f"ERROR: {zip_file} not found on Drive"
print(f"System zip found: {zip_file.name} ({zip_file.stat().st_size/1024:.1f} KB)")

# ── Step 2: Extract and patch topology ───────────────────────────────────────
print("\nExtracting system files...")
with zipfile.ZipFile(zip_file, 'r') as zf:
    zf.extractall(WORK_DIR)

print("Extracted files:")
for f in sorted(WORK_DIR.iterdir()):
    print(f"  {f.name} ({f.stat().st_size/1024:.1f} KB)")

# Patch topology paths
top_file = WORK_DIR / 'topol_complex.top'
top_text = top_file.read_text(encoding='utf-8')

# Show current include lines before patching
print("\nCurrent include lines in topology:")
for line in top_text.splitlines():
    if '#include' in line and ('ligand' in line.lower() or 'posre' in line.lower() or 'CNP' in line):
        print(f"  {line.strip()}")

System zip found: CNP0286261_0_system.zip (208.4 KB)

Extracting system files...
Extracted files:
  CNP0286261_0_GMX.gro (2.7 KB)
  CNP0286261_0_GMX.itp (39.2 KB)
  complex.gro (170.2 KB)
  em.mdp (0.5 KB)
  md.mdp (1.4 KB)
  npt.mdp (1.2 KB)
  nvt.mdp (1.3 KB)
  posre.itp (56.2 KB)
  topol_complex.top (1067.0 KB)

Current include lines in topology:
  #include "../ligand_params/CNP0286261_0/CNP0286261_0_GMX.itp"
  #include "/content/pdb2gmx_work/posre.itp"


In [ ]:
# ── Install GROMACS 2026 via conda and test ───────────────────────────────────

import subprocess, os
from pathlib import Path

WORK_DIR = Path('/content/md_work/CNP0286261_0')
os.chdir(WORK_DIR)

print("Installing GROMACS 2026 via conda (3-5 minutes)...")
r = subprocess.run(
    '/content/miniforge/bin/conda install -y -c conda-forge gromacs -q',
    shell=True, capture_output=True, text=True, timeout=600)
print(f"  Return code: {r.returncode}")
if r.returncode != 0:
    print(r.stderr[-500:])

# Find the installed binary
r2 = subprocess.run('find /content/miniforge -name "gmx*" -type f 2>/dev/null',
                    shell=True, capture_output=True, text=True)
print(f"\nGROMACS binaries after install:\n{r2.stdout}")

# Try running it
for path in r2.stdout.strip().splitlines():
    if 'gmx' in path:
        r3 = subprocess.run(f'{path} --version 2>&1 | head -3',
                            shell=True, capture_output=True, text=True)
        print(f"{path}:\n{r3.stdout}")
        break

Installing GROMACS 2026 via conda (3-5 minutes)...
  Return code: 127
/bin/sh: 1: /content/miniforge/bin/conda: not found


GROMACS binaries after install:



In [ ]:
# ── Fresh install: miniforge + GROMACS 2024 ───────────────────────────────────

import subprocess, os, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

WORK_DIR = Path('/content/md_work/CNP0286261_0')
os.chdir(WORK_DIR)

print("Step 1: Installing miniforge...")
subprocess.run('wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O miniforge.sh', shell=True)
subprocess.run('bash miniforge.sh -b -p /content/miniforge', shell=True, capture_output=True)
print("  Done.")

print("\nStep 2: Installing GROMACS 2024 via conda...")
r = subprocess.run(
    '/content/miniforge/bin/conda install -y -c conda-forge gromacs=2024.4 -q',
    shell=True, capture_output=True, text=True, timeout=600)
if r.returncode != 0:
    # Try without version pin
    print("  2024.4 not found, trying latest...")
    r = subprocess.run(
        '/content/miniforge/bin/conda install -y -c conda-forge gromacs -q',
        shell=True, capture_output=True, text=True, timeout=600)

# Find binary
r2 = subprocess.run('find /content/miniforge -name "gmx" -o -name "gmx_d" 2>/dev/null',
                    shell=True, capture_output=True, text=True)
print(f"\nGROMACS binaries:\n{r2.stdout}")

for path in r2.stdout.strip().splitlines():
    r3 = subprocess.run(f'{path} --version 2>&1 | grep -E "version|GPU"',
                        shell=True, capture_output=True, text=True)
    if r3.stdout:
        print(f"  {path}: {r3.stdout.strip()}")
        GMX_PATH = path
        break

# Test with the problematic system
os.environ['PATH'] = str(Path(GMX_PATH).parent) + ':' + os.environ['PATH']
print(f"\nTesting with CNP0286261_0 system (100 steps)...")
r4 = subprocess.run(
    f'gmx grompp -f md.mdp -c npt.gro -t npt.cpt '
    f'-p topol_complex.top -n index.ndx -o test.tpr -maxwarn 15 && '
    f'gmx mdrun -s test.tpr -deffnm test_run -nsteps 100 -nt 2',
    shell=True, capture_output=True, text=True, timeout=120)

if r4.returncode == 0:
    print("  TEST PASSED — GROMACS 2024 works with this system")
else:
    for line in (r4.stdout + r4.stderr).splitlines():
        if 'Fatal' in line or 'Error' in line or 'Performance' in line:
            print(f"  {line.strip()}")

Mounted at /content/drive
Step 1: Installing miniforge...
  Done.

Step 2: Installing GROMACS 2024 via conda...
  2024.4 not found, trying latest...

GROMACS binaries:
/content/miniforge/bin.SSE2/gmx
/content/miniforge/bin.AVX_256/gmx
/content/miniforge/bin/gmx
/content/miniforge/pkgs/gromacs-2026.0-nompi_h26635d9_101/bin.SSE2/gmx
/content/miniforge/pkgs/gromacs-2026.0-nompi_h26635d9_101/bin.AVX_256/gmx
/content/miniforge/pkgs/gromacs-2026.0-nompi_h26635d9_101/bin/gmx
/content/miniforge/pkgs/gromacs-2026.0-nompi_h26635d9_101/bin.AVX2_256/gmx
/content/miniforge/bin.AVX2_256/gmx

  /content/miniforge/bin.SSE2/gmx: gmx --version
GROMACS version:     2026.0-conda_forge
MPI version:         built in
GPU support:         OpenCL
NBNxM GPU setup:     super-cluster 2x2x2 / cluster 8 (cluster-pair splitting on)
GPU FFT library:     clFFT
Multi-GPU FFT:       none
Colvars support:     enabled (version 2025-10-13)
OpenCL version:      3.0

Testing with CNP0286261_0 system (100 steps)...


In [ ]:
import subprocess, os
from pathlib import Path

WORK_DIR = Path('/content/md_work/CNP0286261_0')
os.chdir(WORK_DIR)
os.environ['PATH'] = '/content/miniforge/bin:' + os.environ['PATH']

print(f"Using gmx: {subprocess.run('which gmx', shell=True, capture_output=True, text=True).stdout.strip()}")

r = subprocess.run(
    'gmx mdrun -s test.tpr -deffnm test_run2 -nsteps 100 -nt 2 -nb cpu -pme cpu',
    shell=True, capture_output=True, text=True, timeout=120)

print(f"Return code: {r.returncode}")
print("\nSTDOUT:")
print(r.stdout[-1000:])
print("\nSTDERR:")
print(r.stderr[-1000:])

Using gmx: /content/miniforge/bin/gmx
Return code: 134

STDOUT:


STDERR:
nt/miniforge
Working dir:  /content/md_work/CNP0286261_0
Command line:
  gmx mdrun -s test.tpr -deffnm test_run2 -nsteps 100 -nt 2 -nb cpu -pme cpu

Compiled SIMD is AVX2_256, but CPU also supports AVX_512 (see log).
The current CPU can measure timings more accurately than the code in
gmx mdrun was configured to use. This might affect your simulation
speed as accurate timings are needed for load-balancing.
Please consider rebuilding gmx mdrun with the GMX_USE_RDTSCP=ON CMake option.
Reading file test.tpr, VERSION 2026.0-conda_forge (single precision)

Overriding nsteps with value passed on the command line: 100 steps, 0.2 ps
Changing nstlist from 20 to 100, rlist from 1 to 1.142


Update groups can not be used for this system because atoms that are (in)directly constrained together are interdispersed with other atoms

Using 1 MPI thread
Using 2 OpenMP threads 

terminate called after throwing an instance of 'std:

In [ ]:
import subprocess, os, shutil
from pathlib import Path

WORK_DIR    = Path('/content/md_work/CNP0286261_0')
DRIVE_BASE  = Path('/content/drive/MyDrive/PfDHFR_MD')
RESULTS_DIR = DRIVE_BASE / 'md_results' / 'CNP0286261_0'
os.chdir(WORK_DIR)
os.environ['PATH'] = '/content/miniforge/bin:' + os.environ['PATH']

# Write MDP with all-bonds constraints instead of h-bonds
print("Writing MDP with modified constraints...")
mdp_content = """; Production MD — all-bonds constraints for GAFF2 ligand compatibility
integrator          = md
nsteps              = 5000000
dt                  = 0.002
nstxout-compressed  = 5000
nstxout             = 0
nstvout             = 0
nstenergy           = 5000
nstlog              = 5000
cutoff-scheme       = Verlet
nstlist             = 20
rcoulomb            = 1.0
rvdw                = 1.0
pbc                 = xyz
coulombtype         = PME
pme_order           = 4
fourierspacing      = 0.12
tcoupl              = V-rescale
tc-grps             = Protein_LIG  Water_and_ions
tau_t               = 0.1          0.1
ref_t               = 310          310
pcoupl              = Parrinello-Rahman
pcoupltype          = isotropic
tau_p               = 2.0
ref_p               = 1.0
compressibility     = 4.5e-5
constraint_algorithm = lincs
constraints         = all-bonds
lincs_iter          = 2
lincs_order         = 6
lincs-warnangle     = 90
gen_vel             = no
DispCorr            = EnerPres
"""
(WORK_DIR / 'md.mdp').write_text(mdp_content)
print("  Done.")

# Also update nvt and npt mdp files
for mdp_name in ['nvt.mdp', 'npt.mdp']:
    mdp_text = (WORK_DIR / mdp_name).read_text(encoding='utf-8')
    mdp_text = mdp_text.replace('constraints         = h-bonds', 'constraints         = all-bonds')
    mdp_text = mdp_text.replace('constraints          = h-bonds', 'constraints         = all-bonds')
    mdp_text += '\nlincs-warnangle     = 90\n'
    (WORK_DIR / mdp_name).write_text(mdp_text, encoding='utf-8')

# Regenerate TPR and test
print("\nRegenerating TPR...")
r = subprocess.run(
    'gmx grompp -f md.mdp -c npt.gro -t npt.cpt '
    '-p topol_complex.top -n index.ndx -o test2.tpr -maxwarn 15',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-500:])
    raise RuntimeError("grompp failed")
print("  test2.tpr created")

print("\nTesting 100 steps...")
r = subprocess.run(
    'gmx mdrun -s test2.tpr -deffnm test_run3 -nsteps 100 -nt 2 -nb cpu -pme cpu',
    shell=True, capture_output=True, text=True, timeout=120)

print(f"Return code: {r.returncode}")
for line in (r.stdout + r.stderr).splitlines():
    if any(x in line for x in ['Performance', 'Fatal', 'Error', 'Abort', 'steps,']):
        print(f"  {line.strip()}")

if r.returncode == 0:
    print("\n  TEST PASSED")
else:
    print("\n  Still failing — checking log...")
    log = WORK_DIR / 'test_run3.log'
    if log.exists():
        print(log.read_text(errors='ignore')[-1000:])

Writing MDP with modified constraints...
  Done.

Regenerating TPR...
  test2.tpr created

Testing 100 steps...
Return code: 134
  Overriding nsteps with value passed on the command line: 100 steps, 0.2 ps
  Aborted (core dumped)

  Still failing — checking log...
nge to:
The maximum number of communication pulses is:
The minimum size for domain decomposition cells is 1.136 nm
The requested allowed shrink of DD cells (option -dds) is: 0.80
The allowed shrink of domain decomposition cells is:
The maximum allowed distance for atoms involved in interactions is:
                 non-bonded interactions           1.136 nm
            two-body bonded interactions  (-rdd)   1.136 nm
          multi-body bonded interactions  (-rdd)   1.136 nm
  atoms separated by up to 7 constraints  (-rcon)  1.136 nm

Local state does not use filler particles

Using 1 MPI thread
Using 2 OpenMP threads 

System total charge: 0.000
Will do PME sum in reciprocal space for electrostatic interactions.

++++ PLEASE

In [ ]:
import subprocess, os, re
from pathlib import Path

WORK_DIR = Path('/content/md_work/CNP0275186_1')
os.chdir(WORK_DIR)
os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']

# Fix extra spaces in molecules section
print("Fixing topology molecules section...")
top_file = WORK_DIR / 'topol_complex.top'
top_text = top_file.read_text(encoding='utf-8')
top_text = re.sub(r'CNP0275186_1\s+1', 'CNP0275186_1        1', top_text)
top_file.write_text(top_text, encoding='utf-8')

last_pos = top_text.rfind('[ molecules ]')
print(top_text[last_pos:last_pos+150])

# Rerun grompp with higher maxwarn
print("\nAdding ions...")
r = subprocess.run(
    'gmx grompp -f em.mdp -c solvated.gro -p topol_complex.top -o ions.tpr -maxwarn 20',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-500:])
    raise RuntimeError("grompp ions failed")

r = subprocess.run(
    'echo "SOL" | gmx genion -s ions.tpr -o ionised.gro '
    '-p topol_complex.top -pname NA -nname CL -neutral -conc 0.15',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-300:])
    raise RuntimeError("genion failed")
for line in (r.stdout + r.stderr).splitlines():
    if 'Will try' in line:
        print(f"  {line.strip()}")

# Build index
print("\nBuilding index...")
r = subprocess.run(['gmx', 'make_ndx', '-f', 'ionised.gro', '-o', 'index.ndx'],
                   input="1 | 13\nname 21 Protein_LIG\nq\n",
                   capture_output=True, text=True)
ndx_path = WORK_DIR / 'index.ndx'
assert ndx_path.exists()
ndx_text = ndx_path.read_text()
if 'Protein_LIG' not in ndx_text:
    ndx_path.write_text(re.sub(r'\[ Protein_UNL \]', '[ Protein_LIG ]', ndx_text))
print(f"  Protein_LIG: {'Protein_LIG' in ndx_path.read_text()}")

# EM
print("\nEnergy minimisation...")
subprocess.run('gmx grompp -f em.mdp -c ionised.gro -p topol_complex.top -o em.tpr -maxwarn 20',
               shell=True, capture_output=True)
r = subprocess.run('gmx mdrun -v -deffnm em -nt 0',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"EM: {r.stderr[-300:]}")
for line in (r.stdout + r.stderr).splitlines():
    if 'converged' in line.lower() or 'Potential Energy' in line:
        print(f"  {line.strip()}")

# Quick GPU test before full NVT
print("\nQuick GPU test (100 steps NVT)...")
r = subprocess.run(
    'gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol_complex.top '
    '-n index.ndx -o nvt_test.tpr -maxwarn 20',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp NVT test: {r.stderr[-500:]}")

r = subprocess.run(
    'gmx mdrun -s nvt_test.tpr -deffnm nvt_test -nsteps 100 '
    '-nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0',
    shell=True, capture_output=True, text=True, timeout=60)

print(f"  GPU test return code: {r.returncode}")
for line in (r.stdout + r.stderr).splitlines():
    if any(x in line for x in ['Performance', 'Fatal', 'Abort', 'steps,']):
        print(f"  {line.strip()}")

if r.returncode == 0:
    print("  GPU test PASSED")
else:
    print("  GPU test failed — check output above")

Fixing topology molecules section...
[ molecules ]
; Compound        #mols
Protein_chain_A     1
CNP0275186_1        1
SOL              3252


Adding ions...
  Will try to add 12 NA ions and 21 CL ions.

Building index...
  Protein_LIG: True

Energy minimisation...
  Steepest Descents converged to Fmax < 1000 in 1026 steps
  Potential Energy  = -1.8539120e+05

Quick GPU test (100 steps NVT)...
  GPU test return code: 0
  Overriding nsteps with value passed on the command line: 100 steps, 0.2 ps
  100 steps,      0.2 ps.
  Performance:       73.337        0.327
  GPU test PASSED


In [ ]:
import subprocess, os, re, shutil, zipfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE    = Path('/content/drive/MyDrive/PfDHFR_MD')
SYSTEMS_DIR   = DRIVE_BASE / 'systems'
RESULTS_DIR   = DRIVE_BASE / 'md_results' / 'CNP0275186_1'
GROMACS_DRIVE = DRIVE_BASE / 'gromacs_cuda_binary'
WORK_DIR      = Path('/content/md_work/CNP0275186_1')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)
os.chdir(WORK_DIR)
MOL_NAME = 'CNP0275186_1'

subprocess.run('apt-get install -y -q libfftw3-dev', shell=True)
os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']
if not Path('/content/gromacs_cuda/bin/gmx').exists():
    shutil.copytree(str(GROMACS_DRIVE), '/content/gromacs_cuda')
    subprocess.run('chmod -R +x /content/gromacs_cuda/bin/', shell=True)
result = subprocess.run('gmx --version', shell=True, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if 'GROMACS version' in line or 'GPU support' in line:
        print(f"  {line.strip()}")

with zipfile.ZipFile(SYSTEMS_DIR / f'{MOL_NAME}_system.zip', 'r') as zf:
    zf.extractall(WORK_DIR)
with open(WORK_DIR / 'complex.gro') as f:
    lines = f.readlines()
box_vals = lines[-1].strip().split()
bx, by, bz = box_vals[0], box_vals[1], box_vals[2]
print(f"  Box: {bx} x {by} x {bz} nm")

top_file = WORK_DIR / 'topol_complex.top'
top_text = top_file.read_text(encoding='utf-8')
top_text = re.sub(r'"\.\./ligand_params/[^"]+\.itp"', f'"{MOL_NAME}_GMX.itp"', top_text)
top_text = re.sub(r'#include\s+"[^"]*posre\.itp"', '#include "posre.itp"', top_text)
top_text = re.sub(r'\bLIG\b(\s+1)', f'{MOL_NAME}\\1', top_text)
top_file.write_text(top_text, encoding='utf-8')

r = subprocess.run(f'gmx solvate -cp complex.gro -cs spc216.gro -o solvated.gro '
                   f'-p topol_complex.top -box {bx} {by} {bz}',
                   shell=True, capture_output=True, text=True)
top_text = top_file.read_text(encoding='utf-8')
top_text = re.sub(f'({re.escape(MOL_NAME)}\\s+1)(SOL)', r'\1\nSOL', top_text)
top_text = re.sub(f'{re.escape(MOL_NAME)}\\s+1', f'{MOL_NAME}        1', top_text)
top_file.write_text(top_text, encoding='utf-8')

r = subprocess.run('gmx grompp -f em.mdp -c solvated.gro -p topol_complex.top -o ions.tpr -maxwarn 20',
                   shell=True, capture_output=True)
subprocess.run('echo "SOL" | gmx genion -s ions.tpr -o ionised.gro '
               '-p topol_complex.top -pname NA -nname CL -neutral -conc 0.15',
               shell=True, capture_output=True)
r = subprocess.run(['gmx', 'make_ndx', '-f', 'ionised.gro', '-o', 'index.ndx'],
                   input="1 | 13\nname 21 Protein_LIG\nq\n", capture_output=True, text=True)
ndx_path = WORK_DIR / 'index.ndx'
ndx_text = ndx_path.read_text()
if 'Protein_LIG' not in ndx_text:
    ndx_path.write_text(re.sub(r'\[ Protein_UNL \]', '[ Protein_LIG ]', ndx_text))

subprocess.run('gmx grompp -f em.mdp -c ionised.gro -p topol_complex.top -o em.tpr -maxwarn 20',
               shell=True, capture_output=True)
r = subprocess.run('gmx mdrun -v -deffnm em -nt 0', shell=True, capture_output=True, text=True)
for line in (r.stdout + r.stderr).splitlines():
    if 'converged' in line.lower() or 'Potential Energy' in line:
        print(f"  {line.strip()}")

r = subprocess.run('gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol_complex.top '
                   '-n index.ndx -o nvt.tpr -maxwarn 20',
                   shell=True, capture_output=True, text=True)
r = subprocess.run('gmx mdrun -deffnm nvt -nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0',
                   shell=True, capture_output=True, text=True, timeout=3600)
for line in (r.stdout + r.stderr).splitlines():
    if 'Performance' in line:
        print(f"  NVT >> {line.strip()}")

with open(WORK_DIR / 'nvt.gro') as f:
    nvt_lines = f.readlines()
print(f"  nvt.gro box: {nvt_lines[-1].strip()}")
print("  Recovery complete — run Cell 2 for NPT + production MD")

Mounted at /content/drive
  GROMACS version:    2023.3
  GPU support:        CUDA
  Box: 4.52550 x 6.38738 x 4.70296 nm
  Steepest Descents converged to Fmax < 1000 in 754 steps
  Potential Energy  = -1.8048016e+05
  NVT >> Performance:       68.825        0.349
  nvt.gro box: 4.52550   6.38738   4.70296
  Recovery complete — run Cell 2 for NPT + production MD


In [ ]:
import subprocess, os, shutil
from pathlib import Path

WORK_DIR    = Path('/content/md_work/CNP0275186_1')
DRIVE_BASE  = Path('/content/drive/MyDrive/PfDHFR_MD')
RESULTS_DIR = DRIVE_BASE / 'md_results' / 'CNP0275186_1'
os.chdir(WORK_DIR)
os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']

# Write corrected npt.mdp with Berendsen barostat
print("Writing corrected npt.mdp (Berendsen barostat)...")
npt_content = """; NPT equilibration — Berendsen barostat for stability
define          = -DPOSRES
integrator      = md
nsteps          = 50000
dt              = 0.002
nstxout         = 500
nstvout         = 500
nstenergy       = 500
nstlog          = 500
cutoff-scheme   = Verlet
nstlist         = 10
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz
coulombtype     = PME
pme_order       = 4
fourierspacing  = 0.16
tcoupl          = V-rescale
tc-grps         = Protein_LIG  Water_and_ions
tau_t           = 0.1          0.1
ref_t           = 310          310
pcoupl              = Berendsen
pcoupltype          = isotropic
tau_p               = 2.0
ref_p               = 1.0
compressibility     = 4.5e-5
refcoord_scaling    = com
constraint_algorithm = lincs
constraints          = h-bonds
lincs_iter           = 1
lincs_order          = 4
gen_vel         = no
"""
(WORK_DIR / 'npt.mdp').write_text(npt_content)

# Run NPT
print("Running NPT with Berendsen barostat...")
r = subprocess.run('gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt '
                   '-p topol_complex.top -n index.ndx -o npt.tpr -maxwarn 20',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp NPT: {r.stderr[-500:]}")
r = subprocess.run('gmx mdrun -deffnm npt -nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0',
                   shell=True, capture_output=True, text=True, timeout=3600)
if r.returncode != 0:
    raise RuntimeError(f"NPT: {r.stderr[-300:]}")
for line in (r.stdout + r.stderr).splitlines():
    if 'Performance' in line:
        print(f"  NPT >> {line.strip()}")

# Verify box did NOT explode
with open(WORK_DIR / 'npt.gro') as f:
    lines = f.readlines()
box = lines[-1].strip()
print(f"  npt.gro box: {box}")
box_vals = [float(x) for x in box.split()]
assert all(v < 10 for v in box_vals[:3]), f"Box exploded: {box}"
print("  Box stable — proceeding to production MD")

# Production MD
print("\nProduction MD — 100 ns in 10 x 10 ns chunks...")
CHUNK_STEPS = 5_000_000
N_CHUNKS    = 10

for chunk in range(1, N_CHUNKS + 1):
    print(f"\n{'─'*60}")
    print(f"  CHUNK {chunk}/{N_CHUNKS}  ({(chunk-1)*10}–{chunk*10} ns)")
    print(f"{'─'*60}")
    out_prefix = f'md_chunk{chunk}'
    prev = 'npt' if chunk == 1 else f'md_chunk{chunk-1}'

    r = subprocess.run(
        f'gmx grompp -f md.mdp -c {prev}.gro -t {prev}.cpt '
        f'-p topol_complex.top -n index.ndx -o {out_prefix}.tpr -maxwarn 20',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1000:])
        raise RuntimeError(f"grompp chunk {chunk} failed.")
    print(f"  {out_prefix}.tpr created")

    r = subprocess.run(
        f'gmx mdrun -deffnm {out_prefix} '
        f'-nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0 -nsteps {CHUNK_STEPS}',
        shell=True, capture_output=True, text=True, timeout=18000)

    for line in (r.stdout + r.stderr).splitlines():
        if 'Performance' in line or 'steps,' in line:
            print(f"  >> {line.strip()}")

    if r.returncode != 0:
        print(r.stderr[-500:])
        raise RuntimeError(f"mdrun chunk {chunk} failed.")

    print(f"  Saving chunk {chunk} to Drive...")
    for ext in ['.xtc', '.gro', '.cpt', '.edr', '.log']:
        src = WORK_DIR / f'{out_prefix}{ext}'
        if src.exists():
            shutil.copy2(src, RESULTS_DIR / src.name)
            print(f"    {src.name} → Drive ({src.stat().st_size/1024/1024:.1f} MB)")
    print(f"  Chunk {chunk} complete.")

print("\n" + "=" * 60)
print("  CNP0275186_1 100 ns MD COMPLETE")
print("  All 10 chunks saved to Drive")
print("  Next: CNP0539885_2")
print("=" * 60)

Writing corrected npt.mdp (Berendsen barostat)...
Running NPT with Berendsen barostat...
  NPT >> Performance:       71.118        0.337
  npt.gro box: 4.49802   6.34859   4.67440
  Box stable — proceeding to production MD

Production MD — 100 ns in 10 x 10 ns chunks...

────────────────────────────────────────────────────────────
  CHUNK 1/10  (0–10 ns)
────────────────────────────────────────────────────────────
  md_chunk1.tpr created
  >> Overriding nsteps with value passed on the command line: 5000000 steps, 1e+04 ps
  >> 5000000 steps,  10000.0 ps.
  >> Performance:       80.887        0.297
  Saving chunk 1 to Drive...
    md_chunk1.xtc → Drive (46.8 MB)
    md_chunk1.gro → Drive (0.9 MB)
    md_chunk1.cpt → Drive (0.3 MB)
    md_chunk1.edr → Drive (0.7 MB)
    md_chunk1.log → Drive (0.6 MB)
  Chunk 1 complete.

────────────────────────────────────────────────────────────
  CHUNK 2/10  (10–20 ns)
────────────────────────────────────────────────────────────
  md_chunk2.tpr create

In [ ]:
# ── CNP0275186_1: Resume from chunk 2 (fixed) ────────────────────────────────

import subprocess, os, re, shutil, zipfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE    = Path('/content/drive/MyDrive/PfDHFR_MD')
SYSTEMS_DIR   = DRIVE_BASE / 'systems'
RESULTS_DIR   = DRIVE_BASE / 'md_results' / 'CNP0275186_1'
GROMACS_DRIVE = DRIVE_BASE / 'gromacs_cuda_binary'
WORK_DIR      = Path('/content/md_work/CNP0275186_1')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)
os.chdir(WORK_DIR)
MOL_NAME = 'CNP0275186_1'

# ── Setup ─────────────────────────────────────────────────────────────────────
print("Step 1: Setup...")
subprocess.run('apt-get install -y -q libfftw3-dev', shell=True)
os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']
if not Path('/content/gromacs_cuda/bin/gmx').exists():
    shutil.copytree(str(GROMACS_DRIVE), '/content/gromacs_cuda')
    subprocess.run('chmod -R +x /content/gromacs_cuda/bin/', shell=True)
result = subprocess.run('gmx --version', shell=True, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if 'GROMACS version' in line or 'GPU support' in line:
        print(f"  {line.strip()}")

# ── Extract and patch ─────────────────────────────────────────────────────────
print("\nStep 2: Extracting and patching...")
with zipfile.ZipFile(SYSTEMS_DIR / f'{MOL_NAME}_system.zip', 'r') as zf:
    zf.extractall(WORK_DIR)
with open(WORK_DIR / 'complex.gro') as f:
    lines = f.readlines()
box_vals = lines[-1].strip().split()
bx, by, bz = box_vals[0], box_vals[1], box_vals[2]
print(f"  Box: {bx} x {by} x {bz} nm")

top_file = WORK_DIR / 'topol_complex.top'
top_text = top_file.read_text(encoding='utf-8')
top_text = re.sub(r'"\.\./ligand_params/[^"]+\.itp"', f'"{MOL_NAME}_GMX.itp"', top_text)
top_text = re.sub(r'#include\s+"[^"]*posre\.itp"', '#include "posre.itp"', top_text)
top_text = re.sub(r'\bLIG\b(\s+1)', f'{MOL_NAME}\\1', top_text)
top_file.write_text(top_text, encoding='utf-8')
print("  Topology patched.")

# ── Solvate ───────────────────────────────────────────────────────────────────
print("\nStep 3: Solvating...")
r = subprocess.run(f'gmx solvate -cp complex.gro -cs spc216.gro -o solvated.gro '
                   f'-p topol_complex.top -box {bx} {by} {bz}',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"solvate failed: {r.stderr[-300:]}")
top_text = top_file.read_text(encoding='utf-8')
top_text = re.sub(f'({re.escape(MOL_NAME)}\\s+1)(SOL)', r'\1\nSOL', top_text)
top_text = re.sub(f'{re.escape(MOL_NAME)}\\s+1', f'{MOL_NAME}        1', top_text)
top_file.write_text(top_text, encoding='utf-8')
with open(WORK_DIR / 'solvated.gro') as f:
    sol_lines = f.readlines()
print(f"  solvated.gro box: {sol_lines[-1].strip()}")
print(f"  solvated.gro atoms: {int(sol_lines[1].strip())}")

# ── Ions ──────────────────────────────────────────────────────────────────────
print("\nStep 4: Adding ions...")
r = subprocess.run('gmx grompp -f em.mdp -c solvated.gro -p topol_complex.top -o ions.tpr -maxwarn 20',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp ions: {r.stderr[-500:]}")
r = subprocess.run('echo "SOL" | gmx genion -s ions.tpr -o ionised.gro '
                   '-p topol_complex.top -pname NA -nname CL -neutral -conc 0.15',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"genion: {r.stderr[-300:]}")
assert (WORK_DIR / 'ionised.gro').exists(), "ionised.gro not created"
for line in (r.stdout + r.stderr).splitlines():
    if 'Will try' in line:
        print(f"  {line.strip()}")

# ── Index ─────────────────────────────────────────────────────────────────────
print("\nStep 5: Building index...")
r = subprocess.run(['gmx', 'make_ndx', '-f', 'ionised.gro', '-o', 'index.ndx'],
                   input="1 | 13\nname 21 Protein_LIG\nq\n",
                   capture_output=True, text=True)
assert (WORK_DIR / 'index.ndx').exists(), f"index.ndx not created: {r.stderr[-200:]}"
ndx_text = (WORK_DIR / 'index.ndx').read_text()
if 'Protein_LIG' not in ndx_text:
    (WORK_DIR / 'index.ndx').write_text(re.sub(r'\[ Protein_UNL \]', '[ Protein_LIG ]', ndx_text))
print(f"  Protein_LIG: {'Protein_LIG' in (WORK_DIR / 'index.ndx').read_text()}")

# ── EM ────────────────────────────────────────────────────────────────────────
print("\nStep 6: Energy minimisation...")
r = subprocess.run('gmx grompp -f em.mdp -c ionised.gro -p topol_complex.top -o em.tpr -maxwarn 20',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp EM: {r.stderr[-300:]}")
r = subprocess.run('gmx mdrun -v -deffnm em -nt 0', shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"EM: {r.stderr[-300:]}")
for line in (r.stdout + r.stderr).splitlines():
    if 'converged' in line.lower() or 'Potential Energy' in line:
        print(f"  {line.strip()}")

# ── NVT ───────────────────────────────────────────────────────────────────────
print("\nStep 7: NVT equilibration...")
r = subprocess.run('gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol_complex.top '
                   '-n index.ndx -o nvt.tpr -maxwarn 20',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp NVT: {r.stderr[-500:]}")
r = subprocess.run('gmx mdrun -deffnm nvt -nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0',
                   shell=True, capture_output=True, text=True, timeout=3600)
if r.returncode != 0:
    raise RuntimeError(f"NVT: {r.stderr[-300:]}")
for line in (r.stdout + r.stderr).splitlines():
    if 'Performance' in line:
        print(f"  NVT >> {line.strip()}")
with open(WORK_DIR / 'nvt.gro') as f:
    nvt_lines = f.readlines()
print(f"  nvt.gro box: {nvt_lines[-1].strip()}")

# ── NPT with Berendsen ────────────────────────────────────────────────────────
print("\nStep 8: NPT equilibration (Berendsen barostat)...")
npt_content = """; NPT — Berendsen barostat for stability
define          = -DPOSRES
integrator      = md
nsteps          = 50000
dt              = 0.002
nstxout         = 500
nstvout         = 500
nstenergy       = 500
nstlog          = 500
cutoff-scheme   = Verlet
nstlist         = 10
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz
coulombtype     = PME
pme_order       = 4
fourierspacing  = 0.16
tcoupl          = V-rescale
tc-grps         = Protein_LIG  Water_and_ions
tau_t           = 0.1          0.1
ref_t           = 310          310
pcoupl              = Berendsen
pcoupltype          = isotropic
tau_p               = 2.0
ref_p               = 1.0
compressibility     = 4.5e-5
refcoord_scaling    = com
constraint_algorithm = lincs
constraints          = h-bonds
lincs_iter           = 1
lincs_order          = 4
gen_vel         = no
"""
(WORK_DIR / 'npt.mdp').write_text(npt_content)
r = subprocess.run('gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt '
                   '-p topol_complex.top -n index.ndx -o npt.tpr -maxwarn 20',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp NPT: {r.stderr[-500:]}")
r = subprocess.run('gmx mdrun -deffnm npt -nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0',
                   shell=True, capture_output=True, text=True, timeout=3600)
if r.returncode != 0:
    raise RuntimeError(f"NPT: {r.stderr[-300:]}")
for line in (r.stdout + r.stderr).splitlines():
    if 'Performance' in line:
        print(f"  NPT >> {line.strip()}")
with open(WORK_DIR / 'npt.gro') as f:
    npt_lines = f.readlines()
npt_box = npt_lines[-1].strip()
print(f"  npt.gro box: {npt_box}")
box_check = [float(x) for x in npt_box.split()]
assert all(v < 10 for v in box_check[:3]), f"Box exploded: {npt_box}"
print("  Box stable.")

# ── Production MD chunks 2-10 ─────────────────────────────────────────────────
print("\nStep 9: Production MD — chunks 2-10...")
CHUNK_STEPS = 5_000_000

# Restore chunk 1 from Drive as starting point
print("  Restoring chunk 1 from Drive...")
for ext in ['.gro', '.cpt']:
    src = RESULTS_DIR / f'md_chunk1{ext}'
    assert src.exists(), f"md_chunk1{ext} not on Drive"
    shutil.copy2(src, WORK_DIR / f'md_chunk1{ext}')
    print(f"    md_chunk1{ext} restored")

for chunk in range(2, 11):
    print(f"\n{'─'*60}")
    print(f"  CHUNK {chunk}/10  ({(chunk-1)*10}–{chunk*10} ns)")
    print(f"{'─'*60}")
    out_prefix = f'md_chunk{chunk}'
    prev = f'md_chunk{chunk-1}'

    r = subprocess.run(
        f'gmx grompp -f md.mdp -c {prev}.gro -t {prev}.cpt '
        f'-p topol_complex.top -n index.ndx -o {out_prefix}.tpr -maxwarn 20',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1000:])
        raise RuntimeError(f"grompp chunk {chunk} failed.")
    print(f"  {out_prefix}.tpr created")

    r = subprocess.run(
        f'gmx mdrun -deffnm {out_prefix} '
        f'-nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0 -nsteps {CHUNK_STEPS}',
        shell=True, capture_output=True, text=True, timeout=36000)

    for line in (r.stdout + r.stderr).splitlines():
        if 'Performance' in line or 'steps,' in line:
            print(f"  >> {line.strip()}")

    if r.returncode != 0:
        print(r.stderr[-500:])
        raise RuntimeError(f"mdrun chunk {chunk} failed.")

    print(f"  Saving chunk {chunk} to Drive...")
    for ext in ['.xtc', '.gro', '.cpt', '.edr', '.log']:
        src = WORK_DIR / f'{out_prefix}{ext}'
        if src.exists():
            shutil.copy2(src, RESULTS_DIR / src.name)
            print(f"    {src.name} → Drive ({src.stat().st_size/1024/1024:.1f} MB)")
    print(f"  Chunk {chunk} complete.")

print("\n" + "=" * 60)
print("  CNP0275186_1 100 ns MD COMPLETE")
print("  All 10 chunks saved to Drive")
print("  Next: CNP0539885_2 (Global NP, -10.03 kcal/mol)")
print("=" * 60)

Mounted at /content/drive
Step 1: Setup...
  GROMACS version:    2023.3
  GPU support:        CUDA

Step 2: Extracting and patching...
  Box: 4.52550 x 6.38738 x 4.70296 nm
  Topology patched.

Step 3: Solvating...
  solvated.gro box: 4.52550   6.38738   4.70296
  solvated.gro atoms: 13549

Step 4: Adding ions...
  Will try to add 12 NA ions and 21 CL ions.

Step 5: Building index...
  Protein_LIG: True

Step 6: Energy minimisation...
  Steepest Descents converged to Fmax < 1000 in 903 steps
  Potential Energy  = -1.8275694e+05

Step 7: NVT equilibration...
  NVT >> Performance:       60.279        0.398
  nvt.gro box: 4.52550   6.38738   4.70296

Step 8: NPT equilibration (Berendsen barostat)...
  NPT >> Performance:       62.272        0.385
  npt.gro box: 4.49097   6.33864   4.66708
  Box stable.

Step 9: Production MD — chunks 2-10...
  Restoring chunk 1 from Drive...
    md_chunk1.gro restored
    md_chunk1.cpt restored

────────────────────────────────────────────────────────────

In [ ]:
# ── CELL: CNP0275186_1 — Auto-resume from last completed chunk ───────────────
# Safe to re-run after any disconnection. Automatically detects last complete
# chunk on Drive and resumes from the next one.

import subprocess, os, re, shutil, zipfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE    = Path('/content/drive/MyDrive/PfDHFR_MD')
SYSTEMS_DIR   = DRIVE_BASE / 'systems'
RESULTS_DIR   = DRIVE_BASE / 'md_results' / 'CNP0275186_1'
GROMACS_DRIVE = DRIVE_BASE / 'gromacs_cuda_binary'
WORK_DIR      = Path('/content/md_work/CNP0275186_1')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MOL_NAME      = 'CNP0275186_1'
CHUNK_STEPS   = 5_000_000
TOTAL_CHUNKS  = 10

# ── Helper: find last fully saved chunk on Drive ──────────────────────────────
def find_last_completed_chunk(results_dir, total_chunks):
    """
    Returns the highest chunk number N for which all 5 output files
    (xtc, gro, cpt, edr, log) exist on Drive. Returns 0 if none complete.
    """
    required_exts = ['.xtc', '.gro', '.cpt', '.edr', '.log']
    last_complete = 0
    for n in range(1, total_chunks + 1):
        all_present = all(
            (results_dir / f'md_chunk{n}{ext}').exists()
            for ext in required_exts
        )
        if all_present:
            last_complete = n
        else:
            break   # chunks must be sequential — stop at first gap
    return last_complete

last_done = find_last_completed_chunk(RESULTS_DIR, TOTAL_CHUNKS)
resume_from = last_done + 1

print("=" * 60)
print(f"  Last complete chunk on Drive : {last_done}")
print(f"  Resuming from chunk          : {resume_from}")
print("=" * 60)

if resume_from > TOTAL_CHUNKS:
    print("  All 10 chunks already complete. Nothing to do.")
    raise SystemExit(0)

# ── Step 1: GROMACS ───────────────────────────────────────────────────────────
print("\nStep 1: GROMACS setup...")
subprocess.run('apt-get install -y -q libfftw3-dev', shell=True)
os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']
if not Path('/content/gromacs_cuda/bin/gmx').exists():
    print("  Copying GROMACS binary from Drive (~2 min)...")
    shutil.copytree(str(GROMACS_DRIVE), '/content/gromacs_cuda')
    subprocess.run('chmod -R +x /content/gromacs_cuda/bin/', shell=True)
r = subprocess.run('gmx --version', shell=True, capture_output=True, text=True)
for line in r.stdout.splitlines():
    if 'GROMACS version' in line or 'GPU support' in line:
        print(f"  {line.strip()}")

# ── Step 2: Workspace ─────────────────────────────────────────────────────────
print("\nStep 2: Preparing workspace...")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)
os.chdir(WORK_DIR)

with zipfile.ZipFile(SYSTEMS_DIR / f'{MOL_NAME}_system.zip', 'r') as zf:
    zf.extractall(WORK_DIR)
with open(WORK_DIR / 'complex.gro') as f:
    lines = f.readlines()
box_vals = lines[-1].strip().split()
bx, by, bz = box_vals[0], box_vals[1], box_vals[2]
print(f"  Box: {bx} x {by} x {bz} nm")

# ── Step 3: Topology patch ────────────────────────────────────────────────────
print("\nStep 3: Patching topology...")
top_file = WORK_DIR / 'topol_complex.top'
top_text = top_file.read_text(encoding='utf-8')
top_text = re.sub(r'"\.\./ligand_params/[^"]+\.itp"', f'"{MOL_NAME}_GMX.itp"', top_text)
top_text = re.sub(r'#include\s+"[^"]*posre\.itp"', '#include "posre.itp"', top_text)
top_text = re.sub(r'\bLIG\b(\s+1)', f'{MOL_NAME}\\1', top_text)
top_file.write_text(top_text, encoding='utf-8')
print("  Topology patched.")

# ── Step 4: Solvate ───────────────────────────────────────────────────────────
print("\nStep 4: Solvating...")
r = subprocess.run(
    f'gmx solvate -cp complex.gro -cs spc216.gro -o solvated.gro '
    f'-p topol_complex.top -box {bx} {by} {bz}',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"Solvate failed:\n{r.stderr[-500:]}")
top_text = top_file.read_text(encoding='utf-8')
top_text = re.sub(f'({re.escape(MOL_NAME)}\\s+1)(SOL)', r'\1\nSOL', top_text)
top_text = re.sub(f'{re.escape(MOL_NAME)}\\s+1', f'{MOL_NAME}        1', top_text)
top_file.write_text(top_text, encoding='utf-8')
with open(WORK_DIR / 'solvated.gro') as f:
    sol_lines = f.readlines()
print(f"  solvated.gro: {int(sol_lines[1].strip())} atoms, box {sol_lines[-1].strip()}")

# ── Step 5: Ions ──────────────────────────────────────────────────────────────
print("\nStep 5: Adding ions...")
r = subprocess.run(
    'gmx grompp -f em.mdp -c solvated.gro -p topol_complex.top -o ions.tpr -maxwarn 20',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp (ions) failed:\n{r.stderr[-500:]}")
r = subprocess.run(
    'echo "SOL" | gmx genion -s ions.tpr -o ionised.gro '
    '-p topol_complex.top -pname NA -nname CL -neutral -conc 0.15',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"genion failed:\n{r.stderr[-300:]}")
for line in (r.stdout + r.stderr).splitlines():
    if 'Will try' in line:
        print(f"  {line.strip()}")

# ── Step 6: Index ─────────────────────────────────────────────────────────────
print("\nStep 6: Building index...")
r = subprocess.run(['gmx', 'make_ndx', '-f', 'ionised.gro', '-o', 'index.ndx'],
                   input="1 | 13\nname 21 Protein_LIG\nq\n",
                   capture_output=True, text=True)
assert (WORK_DIR / 'index.ndx').exists(), \
    f"index.ndx not created.\nstdout: {r.stdout[-300:]}\nstderr: {r.stderr[-300:]}"
ndx_text = (WORK_DIR / 'index.ndx').read_text()
if 'Protein_LIG' not in ndx_text:
    ndx_text = re.sub(r'\[ Protein_UNL \]', '[ Protein_LIG ]', ndx_text)
    (WORK_DIR / 'index.ndx').write_text(ndx_text)
print(f"  Protein_LIG present: {'Protein_LIG' in (WORK_DIR / 'index.ndx').read_text()}")

# ── Step 7: EM ────────────────────────────────────────────────────────────────
print("\nStep 7: Energy minimisation...")
r = subprocess.run(
    'gmx grompp -f em.mdp -c ionised.gro -p topol_complex.top -o em.tpr -maxwarn 20',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp EM failed:\n{r.stderr[-300:]}")
r = subprocess.run('gmx mdrun -v -deffnm em -nt 0',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"EM failed:\n{r.stderr[-300:]}")
for line in (r.stdout + r.stderr).splitlines():
    if 'converged' in line.lower() or 'Potential Energy' in line:
        print(f"  {line.strip()}")

# ── Step 8: NVT ───────────────────────────────────────────────────────────────
print("\nStep 8: NVT equilibration...")
r = subprocess.run(
    'gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol_complex.top '
    '-n index.ndx -o nvt.tpr -maxwarn 20',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp NVT failed:\n{r.stderr[-500:]}")
r = subprocess.run(
    'gmx mdrun -deffnm nvt -nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0',
    shell=True, capture_output=True, text=True, timeout=3600)
if r.returncode != 0:
    raise RuntimeError(f"NVT failed:\n{r.stderr[-300:]}")
for line in (r.stdout + r.stderr).splitlines():
    if 'Performance' in line:
        print(f"  NVT >> {line.strip()}")

# ── Step 9: NPT (Berendsen) ───────────────────────────────────────────────────
print("\nStep 9: NPT equilibration (Berendsen)...")
npt_content = """; NPT equilibration — Berendsen barostat
define          = -DPOSRES
integrator      = md
nsteps          = 50000
dt              = 0.002
nstxout         = 500
nstvout         = 500
nstenergy       = 500
nstlog          = 500
cutoff-scheme   = Verlet
nstlist         = 10
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz
coulombtype     = PME
pme_order       = 4
fourierspacing  = 0.16
tcoupl          = V-rescale
tc-grps         = Protein_LIG  Water_and_ions
tau_t           = 0.1          0.1
ref_t           = 310          310
pcoupl              = Berendsen
pcoupltype          = isotropic
tau_p               = 2.0
ref_p               = 1.0
compressibility     = 4.5e-5
refcoord_scaling    = com
constraint_algorithm = lincs
constraints          = h-bonds
lincs_iter           = 1
lincs_order          = 4
gen_vel         = no
"""
(WORK_DIR / 'npt.mdp').write_text(npt_content)
r = subprocess.run(
    'gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt '
    '-p topol_complex.top -n index.ndx -o npt.tpr -maxwarn 20',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp NPT failed:\n{r.stderr[-500:]}")
r = subprocess.run(
    'gmx mdrun -deffnm npt -nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0',
    shell=True, capture_output=True, text=True, timeout=3600)
if r.returncode != 0:
    raise RuntimeError(f"NPT failed:\n{r.stderr[-300:]}")
for line in (r.stdout + r.stderr).splitlines():
    if 'Performance' in line:
        print(f"  NPT >> {line.strip()}")
with open(WORK_DIR / 'npt.gro') as f:
    npt_lines = f.readlines()
npt_box = npt_lines[-1].strip()
print(f"  npt.gro box: {npt_box}")
assert all(float(v) < 10 for v in npt_box.split()[:3]), f"Box exploded: {npt_box}"
print("  Box stable.")

# ── Step 10: Restore last complete chunk as starting point ────────────────────
print(f"\nStep 10: Restoring chunk {last_done} from Drive as starting point...")
for ext in ['.gro', '.cpt']:
    src = RESULTS_DIR / f'md_chunk{last_done}{ext}'
    if not src.exists():
        raise FileNotFoundError(f"md_chunk{last_done}{ext} not found on Drive")
    shutil.copy2(src, WORK_DIR / f'md_chunk{last_done}{ext}')
    print(f"  md_chunk{last_done}{ext} restored")

# ── Step 11: Production MD chunks ────────────────────────────────────────────
print(f"\nStep 11: Production MD — chunks {resume_from} to {TOTAL_CHUNKS}...")

for chunk in range(resume_from, TOTAL_CHUNKS + 1):
    print(f"\n{'─'*60}")
    print(f"  CHUNK {chunk}/{TOTAL_CHUNKS}  ({(chunk-1)*10}–{chunk*10} ns)")
    print(f"{'─'*60}")
    out_prefix = f'md_chunk{chunk}'
    prev       = f'md_chunk{chunk-1}'

    r = subprocess.run(
        f'gmx grompp -f md.mdp -c {prev}.gro -t {prev}.cpt '
        f'-p topol_complex.top -n index.ndx -o {out_prefix}.tpr -maxwarn 20',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"grompp chunk {chunk} failed:\n{r.stderr[-1000:]}")
    print(f"  {out_prefix}.tpr created")

    r = subprocess.run(
        f'gmx mdrun -deffnm {out_prefix} '
        f'-nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0 -nsteps {CHUNK_STEPS}',
        shell=True, capture_output=True, text=True, timeout=36000)

    for line in (r.stdout + r.stderr).splitlines():
        if 'Performance' in line or 'steps,' in line:
            print(f"  >> {line.strip()}")

    if r.returncode != 0:
        raise RuntimeError(f"mdrun chunk {chunk} failed:\n{r.stderr[-800:]}")

    print(f"  Saving chunk {chunk} to Drive...")
    for ext in ['.xtc', '.gro', '.cpt', '.edr', '.log']:
        src = WORK_DIR / f'{out_prefix}{ext}'
        if src.exists():
            shutil.copy2(src, RESULTS_DIR / src.name)
            print(f"    {src.name} → Drive ({src.stat().st_size/1024/1024:.1f} MB)")
    print(f"  Chunk {chunk} complete.")

print("\n" + "=" * 60)
print(f"  CNP0275186_1 — 100 ns MD COMPLETE")
print(f"  All {TOTAL_CHUNKS} chunks saved to Drive")
print(f"  Next: CNP0539885_2 (Global NP, -10.03 kcal/mol)")
print("=" * 60)

Mounted at /content/drive
  Last complete chunk on Drive : 4
  Resuming from chunk          : 5

Step 1: GROMACS setup...
  Copying GROMACS binary from Drive (~2 min)...
  GROMACS version:    2023.3
  GPU support:        CUDA

Step 2: Preparing workspace...
  Box: 4.52550 x 6.38738 x 4.70296 nm

Step 3: Patching topology...
  Topology patched.

Step 4: Solvating...
  solvated.gro: 13549 atoms, box 4.52550   6.38738   4.70296

Step 5: Adding ions...
  Will try to add 12 NA ions and 21 CL ions.

Step 6: Building index...
  Protein_LIG present: True

Step 7: Energy minimisation...
  Steepest Descents converged to Fmax < 1000 in 658 steps
  Potential Energy  = -1.7827620e+05

Step 8: NVT equilibration...
  NVT >> Performance:       63.497        0.378

Step 9: NPT equilibration (Berendsen)...
  NPT >> Performance:       72.957        0.329
  npt.gro box: 4.49162   6.33957   4.66775
  Box stable.

Step 10: Restoring chunk 4 from Drive as starting point...
  md_chunk4.gro restored
  md_chunk

In [ ]:
# ── CELL: CNP0539885_2 — Full pipeline + auto-resume ────────────────────────

import subprocess, os, re, shutil, zipfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE    = Path('/content/drive/MyDrive/PfDHFR_MD')
SYSTEMS_DIR   = DRIVE_BASE / 'systems'
RESULTS_DIR   = DRIVE_BASE / 'md_results' / 'CNP0539885_2'
GROMACS_DRIVE = DRIVE_BASE / 'gromacs_cuda_binary'
WORK_DIR      = Path('/content/md_work/CNP0539885_2')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MOL_NAME      = 'CNP0539885_2'
CHUNK_STEPS   = 5_000_000
TOTAL_CHUNKS  = 10

# ── Find last complete chunk on Drive ─────────────────────────────────────────
def find_last_completed_chunk(results_dir, total_chunks):
    required_exts = ['.xtc', '.gro', '.cpt', '.edr', '.log']
    last_complete = 0
    for n in range(1, total_chunks + 1):
        if all((results_dir / f'md_chunk{n}{ext}').exists() for ext in required_exts):
            last_complete = n
        else:
            break
    return last_complete

last_done   = find_last_completed_chunk(RESULTS_DIR, TOTAL_CHUNKS)
resume_from = last_done + 1
print("=" * 60)
print(f"  Last complete chunk on Drive : {last_done}")
print(f"  Resuming from chunk          : {resume_from}")
print("=" * 60)
if resume_from > TOTAL_CHUNKS:
    print("  All 10 chunks already complete. Nothing to do.")
    raise SystemExit(0)

# ── Step 1: GROMACS ───────────────────────────────────────────────────────────
print("\nStep 1: GROMACS setup...")
subprocess.run('apt-get install -y -q libfftw3-dev', shell=True)
os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']
if not Path('/content/gromacs_cuda/bin/gmx').exists():
    print("  Copying GROMACS binary from Drive (~2 min)...")
    shutil.copytree(str(GROMACS_DRIVE), '/content/gromacs_cuda')
    subprocess.run('chmod -R +x /content/gromacs_cuda/bin/', shell=True)
r = subprocess.run('gmx --version', shell=True, capture_output=True, text=True)
for line in r.stdout.splitlines():
    if 'GROMACS version' in line or 'GPU support' in line:
        print(f"  {line.strip()}")

# ── Step 2: Workspace ─────────────────────────────────────────────────────────
print("\nStep 2: Preparing workspace...")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)
os.chdir(WORK_DIR)
with zipfile.ZipFile(SYSTEMS_DIR / f'{MOL_NAME}_system.zip', 'r') as zf:
    zf.extractall(WORK_DIR)
with open(WORK_DIR / 'complex.gro') as f:
    lines = f.readlines()
box_vals = lines[-1].strip().split()
bx, by, bz = box_vals[0], box_vals[1], box_vals[2]
print(f"  Box: {bx} x {by} x {bz} nm")

# ── Step 3: Read mol_name from ITP ────────────────────────────────────────────
print("\nStep 3: Reading molecule name from ITP...")
itp_file = WORK_DIR / f'{MOL_NAME}_GMX.itp'
itp_text = itp_file.read_text(encoding='utf-8')
mol_name_itp = None
in_moltype = False
for line in itp_text.splitlines():
    if '[ moleculetype ]' in line:
        in_moltype = True
        continue
    if in_moltype and line.startswith(';'):
        continue
    if in_moltype and line.strip():
        mol_name_itp = line.strip().split()[0]
        break
print(f"  ITP molecule name: {mol_name_itp}")
assert mol_name_itp is not None, "Could not read mol_name from ITP"

# ── Step 4: Patch topology ────────────────────────────────────────────────────
print("\nStep 4: Patching topology...")
top_file = WORK_DIR / 'topol_complex.top'
top_text = top_file.read_text(encoding='utf-8')
top_text = re.sub(r'"\.\./ligand_params/[^"]+\.itp"', f'"{MOL_NAME}_GMX.itp"', top_text)
top_text = re.sub(r'#include\s+"[^"]*posre\.itp"', '#include "posre.itp"', top_text)
top_text = re.sub(r'\bLIG\b(\s+1)', f'{mol_name_itp}\\1', top_text)
top_file.write_text(top_text, encoding='utf-8')
print("  Topology patched.")

# ── Step 5: Solvate ───────────────────────────────────────────────────────────
print("\nStep 5: Solvating...")
r = subprocess.run(
    f'gmx solvate -cp complex.gro -cs spc216.gro -o solvated.gro '
    f'-p topol_complex.top -box {bx} {by} {bz}',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"Solvate failed:\n{r.stderr[-500:]}")
top_text = top_file.read_text(encoding='utf-8')
top_text = re.sub(f'({re.escape(mol_name_itp)}\\s+1)(SOL)', r'\1\nSOL', top_text)
top_text = re.sub(f'{re.escape(mol_name_itp)}\\s+1', f'{mol_name_itp}        1', top_text)
top_file.write_text(top_text, encoding='utf-8')
with open(WORK_DIR / 'solvated.gro') as f:
    sol_lines = f.readlines()
print(f"  solvated.gro: {int(sol_lines[1].strip())} atoms, box {sol_lines[-1].strip()}")

# ── Step 6: Ions ──────────────────────────────────────────────────────────────
print("\nStep 6: Adding ions...")
r = subprocess.run(
    'gmx grompp -f em.mdp -c solvated.gro -p topol_complex.top -o ions.tpr -maxwarn 20',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp (ions) failed:\n{r.stderr[-500:]}")
r = subprocess.run(
    'echo "SOL" | gmx genion -s ions.tpr -o ionised.gro '
    '-p topol_complex.top -pname NA -nname CL -neutral -conc 0.15',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"genion failed:\n{r.stderr[-300:]}")
assert (WORK_DIR / 'ionised.gro').exists(), "ionised.gro not created"
for line in (r.stdout + r.stderr).splitlines():
    if 'Will try' in line:
        print(f"  {line.strip()}")

# ── Step 7: Index ─────────────────────────────────────────────────────────────
print("\nStep 7: Building index...")
r = subprocess.run(
    ['gmx', 'make_ndx', '-f', 'ionised.gro', '-o', 'index.ndx'],
    input="1 | 13\nname 21 Protein_LIG\nq\n",
    capture_output=True, text=True)
assert (WORK_DIR / 'index.ndx').exists(), \
    f"index.ndx not created.\nstderr: {r.stderr[-300:]}"
ndx_text = (WORK_DIR / 'index.ndx').read_text()
if 'Protein_LIG' not in ndx_text:
    ndx_text = re.sub(r'\[ Protein_UNL \]', '[ Protein_LIG ]', ndx_text)
    (WORK_DIR / 'index.ndx').write_text(ndx_text)
print(f"  Protein_LIG present: {'Protein_LIG' in (WORK_DIR / 'index.ndx').read_text()}")

# ── Step 8: Energy minimisation ───────────────────────────────────────────────
print("\nStep 8: Energy minimisation...")
r = subprocess.run(
    'gmx grompp -f em.mdp -c ionised.gro -p topol_complex.top -o em.tpr -maxwarn 20',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp EM failed:\n{r.stderr[-300:]}")
r = subprocess.run('gmx mdrun -v -deffnm em -nt 0',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"EM failed:\n{r.stderr[-300:]}")
for line in (r.stdout + r.stderr).splitlines():
    if 'converged' in line.lower() or 'Potential Energy' in line:
        print(f"  {line.strip()}")

# ── Step 9: NVT ───────────────────────────────────────────────────────────────
print("\nStep 9: NVT equilibration...")
r = subprocess.run(
    'gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol_complex.top '
    '-n index.ndx -o nvt.tpr -maxwarn 20',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp NVT failed:\n{r.stderr[-500:]}")
r = subprocess.run(
    'gmx mdrun -deffnm nvt -nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0',
    shell=True, capture_output=True, text=True, timeout=3600)
if r.returncode != 0:
    raise RuntimeError(f"NVT failed:\n{r.stderr[-300:]}")
for line in (r.stdout + r.stderr).splitlines():
    if 'Performance' in line:
        print(f"  NVT >> {line.strip()}")

# ── Step 10: NPT (Berendsen) ──────────────────────────────────────────────────
print("\nStep 10: NPT equilibration (Berendsen)...")
npt_content = """; NPT equilibration - Berendsen barostat
define          = -DPOSRES
integrator      = md
nsteps          = 50000
dt              = 0.002
nstxout         = 500
nstvout         = 500
nstenergy       = 500
nstlog          = 500
cutoff-scheme   = Verlet
nstlist         = 10
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz
coulombtype     = PME
pme_order       = 4
fourierspacing  = 0.16
tcoupl          = V-rescale
tc-grps         = Protein_LIG  Water_and_ions
tau_t           = 0.1          0.1
ref_t           = 310          310
pcoupl              = Berendsen
pcoupltype          = isotropic
tau_p               = 2.0
ref_p               = 1.0
compressibility     = 4.5e-5
refcoord_scaling    = com
constraint_algorithm = lincs
constraints          = h-bonds
lincs_iter           = 1
lincs_order          = 4
gen_vel         = no
"""
(WORK_DIR / 'npt.mdp').write_text(npt_content)
r = subprocess.run(
    'gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt '
    '-p topol_complex.top -n index.ndx -o npt.tpr -maxwarn 20',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp NPT failed:\n{r.stderr[-500:]}")
r = subprocess.run(
    'gmx mdrun -deffnm npt -nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0',
    shell=True, capture_output=True, text=True, timeout=3600)
if r.returncode != 0:
    raise RuntimeError(f"NPT failed:\n{r.stderr[-300:]}")
for line in (r.stdout + r.stderr).splitlines():
    if 'Performance' in line:
        print(f"  NPT >> {line.strip()}")
with open(WORK_DIR / 'npt.gro') as f:
    npt_lines = f.readlines()
npt_box = npt_lines[-1].strip()
print(f"  npt.gro box: {npt_box}")
assert all(float(v) < 10 for v in npt_box.split()[:3]), f"Box exploded: {npt_box}"
print("  Box stable.")

# ── Step 11: Production MD (from chunk 1 or resume) ───────────────────────────
if last_done == 0:
    start_gro = 'npt.gro'
    start_cpt = 'npt.cpt'
    print(f"\nStep 11: Production MD - starting from chunk 1...")
else:
    print(f"\nStep 11: Restoring chunk {last_done} from Drive...")
    for ext in ['.gro', '.cpt']:
        src = RESULTS_DIR / f'md_chunk{last_done}{ext}'
        if not src.exists():
            raise FileNotFoundError(f"md_chunk{last_done}{ext} not found on Drive")
        shutil.copy2(src, WORK_DIR / f'md_chunk{last_done}{ext}')
        print(f"  md_chunk{last_done}{ext} restored")
    print(f"  Starting from chunk {resume_from}...")

print(f"\nProduction MD - chunks {resume_from} to {TOTAL_CHUNKS}...")

for chunk in range(resume_from, TOTAL_CHUNKS + 1):
    print(f"\n{'─'*60}")
    print(f"  CHUNK {chunk}/{TOTAL_CHUNKS}  ({(chunk-1)*10}-{chunk*10} ns)")
    print(f"{'─'*60}")
    out_prefix = f'md_chunk{chunk}'

    if chunk == 1:
        prev_gro = 'npt.gro'
        prev_cpt = 'npt.cpt'
        grompp_cmd = (
            f'gmx grompp -f md.mdp -c {prev_gro} -t {prev_cpt} '
            f'-p topol_complex.top -n index.ndx -o {out_prefix}.tpr -maxwarn 20'
        )
    else:
        prev = f'md_chunk{chunk-1}'
        grompp_cmd = (
            f'gmx grompp -f md.mdp -c {prev}.gro -t {prev}.cpt '
            f'-p topol_complex.top -n index.ndx -o {out_prefix}.tpr -maxwarn 20'
        )

    r = subprocess.run(grompp_cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"grompp chunk {chunk} failed:\n{r.stderr[-1000:]}")
    print(f"  {out_prefix}.tpr created")

    r = subprocess.run(
        f'gmx mdrun -deffnm {out_prefix} '
        f'-nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0 -nsteps {CHUNK_STEPS}',
        shell=True, capture_output=True, text=True, timeout=36000)

    for line in (r.stdout + r.stderr).splitlines():
        if 'Performance' in line or 'steps,' in line:
            print(f"  >> {line.strip()}")

    if r.returncode != 0:
        raise RuntimeError(f"mdrun chunk {chunk} failed:\n{r.stderr[-800:]}")

    print(f"  Saving chunk {chunk} to Drive...")
    for ext in ['.xtc', '.gro', '.cpt', '.edr', '.log']:
        src = WORK_DIR / f'{out_prefix}{ext}'
        if src.exists():
            shutil.copy2(src, RESULTS_DIR / src.name)
            print(f"    {src.name} -> Drive ({src.stat().st_size/1024/1024:.1f} MB)")
    print(f"  Chunk {chunk} complete.")

print("\n" + "=" * 60)
print(f"  CNP0539885_2 - 100 ns MD COMPLETE")
print(f"  All {TOTAL_CHUNKS} chunks saved to Drive")
print("  Next: Notebook 05c (local analysis)")
print("=" * 60)

Mounted at /content/drive
  Last complete chunk on Drive : 8
  Resuming from chunk          : 9

Step 1: GROMACS setup...
  Copying GROMACS binary from Drive (~2 min)...
  GROMACS version:    2023.3
  GPU support:        CUDA

Step 2: Preparing workspace...
  Box: 4.52550 x 6.38738 x 4.70296 nm

Step 3: Reading molecule name from ITP...
  ITP molecule name: CNP0539885_2

Step 4: Patching topology...
  Topology patched.

Step 5: Solvating...
  solvated.gro: 13550 atoms, box 4.52550   6.38738   4.70296

Step 6: Adding ions...
  Will try to add 12 NA ions and 21 CL ions.

Step 7: Building index...
  Protein_LIG present: True

Step 8: Energy minimisation...
  Steepest Descents converged to Fmax < 1000 in 725 steps
  Potential Energy  = -1.8400966e+05

Step 9: NVT equilibration...
  NVT >> Performance:       55.821        0.430

Step 10: NPT equilibration (Berendsen)...
  NPT >> Performance:       64.119        0.374
  npt.gro box: 4.49572   6.34535   4.67202
  Box stable.

Step 11: Restori

In [ ]:
# ── CELL: Fix broken GROMACS copy + resume CNP0286261_0 ─────────────────────

import subprocess, os, re, shutil, zipfile, time
from pathlib import Path
from google.colab import drive

# Step 0: Clean up broken partial copy and remount Drive
print("Step 0: Cleaning up and remounting Drive...")
if Path('/content/gromacs_cuda').exists():
    shutil.rmtree('/content/gromacs_cuda')
    print("  Removed partial gromacs_cuda copy")

drive.mount('/content/drive', force_remount=True)
time.sleep(5)  # let mount stabilise

DRIVE_BASE    = Path('/content/drive/MyDrive/PfDHFR_MD')
SYSTEMS_DIR   = DRIVE_BASE / 'systems'
RESULTS_DIR   = DRIVE_BASE / 'md_results' / 'CNP0286261_0'
GROMACS_DRIVE = DRIVE_BASE / 'gromacs_cuda_binary'
WORK_DIR      = Path('/content/md_work/CNP0286261_0')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MOL_NAME      = 'CNP0286261_0'
CHUNK_STEPS   = 5_000_000
TOTAL_CHUNKS  = 10

# Verify Drive is actually accessible before proceeding
assert GROMACS_DRIVE.exists(), f"Drive not accessible: {GROMACS_DRIVE}"
print(f"  Drive mounted OK. GROMACS binary dir exists: {GROMACS_DRIVE.exists()}")

# ── Find last complete chunk ───────────────────────────────────────────────────
def find_last_completed_chunk(results_dir, total_chunks):
    required_exts = ['.xtc', '.gro', '.cpt', '.edr', '.log']
    last_complete = 0
    for n in range(1, total_chunks + 1):
        if all((results_dir / f'md_chunk{n}{ext}').exists() for ext in required_exts):
            last_complete = n
        else:
            break
    return last_complete

last_done   = find_last_completed_chunk(RESULTS_DIR, TOTAL_CHUNKS)
resume_from = last_done + 1
print("=" * 60)
print(f"  Last complete chunk on Drive : {last_done}")
print(f"  Resuming from chunk          : {resume_from}")
print("=" * 60)
if resume_from > TOTAL_CHUNKS:
    print("  All 10 chunks already complete. Nothing to do.")
    raise SystemExit(0)

# ── Step 1: GROMACS ───────────────────────────────────────────────────────────
print("\nStep 1: GROMACS setup...")
subprocess.run('apt-get install -y -q libfftw3-dev', shell=True)
os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']
print("  Copying GROMACS binary from Drive (~2 min)...")
shutil.copytree(str(GROMACS_DRIVE), '/content/gromacs_cuda')
subprocess.run('chmod -R +x /content/gromacs_cuda/bin/', shell=True)
time.sleep(10)
r = subprocess.run('gmx --version', shell=True, capture_output=True, text=True)
for line in r.stdout.splitlines():
    if 'GROMACS version' in line or 'GPU support' in line:
        print(f"  {line.strip()}")
assert 'CUDA' in r.stdout, "GROMACS CUDA not loaded correctly"

# ── Step 2: Workspace ─────────────────────────────────────────────────────────
print("\nStep 2: Preparing workspace...")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)
os.chdir(WORK_DIR)
with zipfile.ZipFile(SYSTEMS_DIR / f'{MOL_NAME}_system.zip', 'r') as zf:
    zf.extractall(WORK_DIR)
with open(WORK_DIR / 'complex.gro') as f:
    lines = f.readlines()
box_vals = lines[-1].strip().split()
bx, by, bz = box_vals[0], box_vals[1], box_vals[2]
print(f"  Box: {bx} x {by} x {bz} nm")

# ── Step 3: Read mol_name from ITP ────────────────────────────────────────────
print("\nStep 3: Reading molecule name from ITP...")
itp_file = WORK_DIR / f'{MOL_NAME}_GMX.itp'
itp_text = itp_file.read_text(encoding='utf-8')
mol_name_itp = None
in_moltype = False
for line in itp_text.splitlines():
    if '[ moleculetype ]' in line:
        in_moltype = True
        continue
    if in_moltype and line.startswith(';'):
        continue
    if in_moltype and line.strip():
        mol_name_itp = line.strip().split()[0]
        break
print(f"  ITP molecule name: {mol_name_itp}")
assert mol_name_itp is not None

# ── Step 4: Patch topology ────────────────────────────────────────────────────
print("\nStep 4: Patching topology...")
top_file = WORK_DIR / 'topol_complex.top'
top_text = top_file.read_text(encoding='utf-8')
top_text = re.sub(r'"\.\./ligand_params/[^"]+\.itp"', f'"{MOL_NAME}_GMX.itp"', top_text)
top_text = re.sub(r'#include\s+"[^"]*posre\.itp"', '#include "posre.itp"', top_text)
top_text = re.sub(r'\bLIG\b(\s+1)', f'{mol_name_itp}\\1', top_text)
top_file.write_text(top_text, encoding='utf-8')
print("  Topology patched.")

# ── Step 5: Solvate ───────────────────────────────────────────────────────────
print("\nStep 5: Solvating...")
r = subprocess.run(
    f'gmx solvate -cp complex.gro -cs spc216.gro -o solvated.gro '
    f'-p topol_complex.top -box {bx} {by} {bz}',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"Solvate failed:\n{r.stderr[-500:]}")
top_text = top_file.read_text(encoding='utf-8')
top_text = re.sub(f'({re.escape(mol_name_itp)}\\s+1)(SOL)', r'\1\nSOL', top_text)
top_text = re.sub(f'{re.escape(mol_name_itp)}\\s+1', f'{mol_name_itp}        1', top_text)
top_file.write_text(top_text, encoding='utf-8')
with open(WORK_DIR / 'solvated.gro') as f:
    sol_lines = f.readlines()
print(f"  solvated.gro: {int(sol_lines[1].strip())} atoms, box {sol_lines[-1].strip()}")

# ── Step 6: Ions ──────────────────────────────────────────────────────────────
print("\nStep 6: Adding ions...")
r = subprocess.run(
    'gmx grompp -f em.mdp -c solvated.gro -p topol_complex.top -o ions.tpr -maxwarn 20',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp (ions) failed:\n{r.stderr[-500:]}")
r = subprocess.run(
    'echo "SOL" | gmx genion -s ions.tpr -o ionised.gro '
    '-p topol_complex.top -pname NA -nname CL -neutral -conc 0.15',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"genion failed:\n{r.stderr[-300:]}")
assert (WORK_DIR / 'ionised.gro').exists(), "ionised.gro not created"
for line in (r.stdout + r.stderr).splitlines():
    if 'Will try' in line:
        print(f"  {line.strip()}")

# ── Step 7: Index ─────────────────────────────────────────────────────────────
print("\nStep 7: Building index...")
r = subprocess.run(
    ['gmx', 'make_ndx', '-f', 'ionised.gro', '-o', 'index.ndx'],
    input="1 | 13\nname 21 Protein_LIG\nq\n",
    capture_output=True, text=True)
assert (WORK_DIR / 'index.ndx').exists(), \
    f"index.ndx not created.\nstderr: {r.stderr[-300:]}"
ndx_text = (WORK_DIR / 'index.ndx').read_text()
if 'Protein_LIG' not in ndx_text:
    ndx_text = re.sub(r'\[ Protein_UNL \]', '[ Protein_LIG ]', ndx_text)
    (WORK_DIR / 'index.ndx').write_text(ndx_text)
print(f"  Protein_LIG present: {'Protein_LIG' in (WORK_DIR / 'index.ndx').read_text()}")

# ── Step 8: Energy minimisation ───────────────────────────────────────────────
print("\nStep 8: Energy minimisation...")
r = subprocess.run(
    'gmx grompp -f em.mdp -c ionised.gro -p topol_complex.top -o em.tpr -maxwarn 20',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp EM failed:\n{r.stderr[-500:]}")
r = subprocess.run('gmx mdrun -v -deffnm em -nt 0',
                   shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"EM failed:\n{r.stderr[-300:]}")
for line in (r.stdout + r.stderr).splitlines():
    if 'converged' in line.lower() or 'Potential Energy' in line:
        print(f"  {line.strip()}")

# ── Step 9: NVT ───────────────────────────────────────────────────────────────
print("\nStep 9: NVT equilibration...")
r = subprocess.run(
    'gmx grompp -f nvt.mdp -c em.gro -r em.gro -p topol_complex.top '
    '-n index.ndx -o nvt.tpr -maxwarn 20',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp NVT failed:\n{r.stderr[-500:]}")
r = subprocess.run(
    'gmx mdrun -deffnm nvt -nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0',
    shell=True, capture_output=True, text=True, timeout=3600)
if r.returncode != 0:
    raise RuntimeError(f"NVT failed:\n{r.stderr[-300:]}")
for line in (r.stdout + r.stderr).splitlines():
    if 'Performance' in line:
        print(f"  NVT >> {line.strip()}")

# ── Step 10: NPT (Berendsen) ──────────────────────────────────────────────────
print("\nStep 10: NPT equilibration (Berendsen)...")
npt_content = """; NPT equilibration - Berendsen barostat
define          = -DPOSRES
integrator      = md
nsteps          = 50000
dt              = 0.002
nstxout         = 500
nstvout         = 500
nstenergy       = 500
nstlog          = 500
cutoff-scheme   = Verlet
nstlist         = 10
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz
coulombtype     = PME
pme_order       = 4
fourierspacing  = 0.16
tcoupl          = V-rescale
tc-grps         = Protein_LIG  Water_and_ions
tau_t           = 0.1          0.1
ref_t           = 310          310
pcoupl              = Berendsen
pcoupltype          = isotropic
tau_p               = 2.0
ref_p               = 1.0
compressibility     = 4.5e-5
refcoord_scaling    = com
constraint_algorithm = lincs
constraints          = h-bonds
lincs_iter           = 1
lincs_order          = 4
gen_vel         = no
"""
(WORK_DIR / 'npt.mdp').write_text(npt_content)
r = subprocess.run(
    'gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt '
    '-p topol_complex.top -n index.ndx -o npt.tpr -maxwarn 20',
    shell=True, capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"grompp NPT failed:\n{r.stderr[-500:]}")
r = subprocess.run(
    'gmx mdrun -deffnm npt -nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0',
    shell=True, capture_output=True, text=True, timeout=3600)
if r.returncode != 0:
    raise RuntimeError(f"NPT failed:\n{r.stderr[-300:]}")
for line in (r.stdout + r.stderr).splitlines():
    if 'Performance' in line:
        print(f"  NPT >> {line.strip()}")
with open(WORK_DIR / 'npt.gro') as f:
    npt_lines = f.readlines()
npt_box = npt_lines[-1].strip()
print(f"  npt.gro box: {npt_box}")
assert all(float(v) < 10 for v in npt_box.split()[:3]), f"Box exploded: {npt_box}"
print("  Box stable.")

# ── Step 11: Production MD ────────────────────────────────────────────────────
print(f"\nStep 11: Restoring chunk {last_done} from Drive...")
for ext in ['.gro', '.cpt']:
    src = RESULTS_DIR / f'md_chunk{last_done}{ext}'
    if not src.exists():
        raise FileNotFoundError(f"md_chunk{last_done}{ext} not found on Drive")
    shutil.copy2(src, WORK_DIR / f'md_chunk{last_done}{ext}')
    print(f"  md_chunk{last_done}{ext} restored")

print(f"\nProduction MD - chunks {resume_from} to {TOTAL_CHUNKS}...")

for chunk in range(resume_from, TOTAL_CHUNKS + 1):
    print(f"\n{'─'*60}")
    print(f"  CHUNK {chunk}/{TOTAL_CHUNKS}  ({(chunk-1)*10}-{chunk*10} ns)")
    print(f"{'─'*60}")
    out_prefix = f'md_chunk{chunk}'
    prev       = f'md_chunk{chunk-1}'

    r = subprocess.run(
        f'gmx grompp -f md.mdp -c {prev}.gro -t {prev}.cpt '
        f'-p topol_complex.top -n index.ndx -o {out_prefix}.tpr -maxwarn 20',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"grompp chunk {chunk} failed:\n{r.stderr[-1000:]}")
    print(f"  {out_prefix}.tpr created")

    r = subprocess.run(
        f'gmx mdrun -deffnm {out_prefix} '
        f'-nb gpu -pme cpu -update cpu -gpu_id 0 -nt 0 -nsteps {CHUNK_STEPS}',
        shell=True, capture_output=True, text=True, timeout=36000)

    for line in (r.stdout + r.stderr).splitlines():
        if 'Performance' in line or 'steps,' in line:
            print(f"  >> {line.strip()}")

    if r.returncode != 0:
        raise RuntimeError(f"mdrun chunk {chunk} failed:\n{r.stderr[-800:]}")

    print(f"  Saving chunk {chunk} to Drive...")
    for ext in ['.xtc', '.gro', '.cpt', '.edr', '.log']:
        src = WORK_DIR / f'{out_prefix}{ext}'
        if src.exists():
            shutil.copy2(src, RESULTS_DIR / src.name)
            print(f"    {src.name} -> Drive ({src.stat().st_size/1024/1024:.1f} MB)")
    print(f"  Chunk {chunk} complete.")

print("\n" + "=" * 60)
print("  CNP0286261_0 - 100 ns MD COMPLETE")
print("  Notebook 05b FULLY COMPLETE")
print("  Next: Notebook 05c (local analysis)")
print("=" * 60)

Step 0: Cleaning up and remounting Drive...
Mounted at /content/drive
  Drive mounted OK. GROMACS binary dir exists: True
  Last complete chunk on Drive : 9
  Resuming from chunk          : 10

Step 1: GROMACS setup...
  Copying GROMACS binary from Drive (~2 min)...
  GROMACS version:    2023.3
  GPU support:        CUDA

Step 2: Preparing workspace...
  Box: 4.52550 x 6.38738 x 4.70296 nm

Step 3: Reading molecule name from ITP...
  ITP molecule name: CNP0286261_0

Step 4: Patching topology...
  Topology patched.

Step 5: Solvating...
  solvated.gro: 13553 atoms, box 4.52550   6.38738   4.70296

Step 6: Adding ions...
  Will try to add 12 NA ions and 21 CL ions.

Step 7: Building index...
  Protein_LIG present: True

Step 8: Energy minimisation...
  Steepest Descents converged to Fmax < 1000 in 792 steps
  Potential Energy  = -1.8422383e+05

Step 9: NVT equilibration...
  NVT >> Performance:       74.766        0.321

Step 10: NPT equilibration (Berendsen)...
  NPT >> Performance:    

In [ ]:
# ── Verify all MD output files on Drive ──────────────────────────────────────
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/PfDHFR_MD/md_results')
compounds  = {
    'pyrimethamine': 5,
    'CNP0275186_1' : 10,
    'CNP0539885_2' : 10,
    'CNP0286261_0' : 10,
}
required_exts = ['.xtc', '.gro', '.cpt', '.edr', '.log']

print("=" * 60)
all_ok = True
for compound, total in compounds.items():
    print(f"\n  {compound} ({total} chunks expected):")
    for chunk in range(1, total + 1):
        files   = [DRIVE_BASE / compound / f'md_chunk{chunk}{ext}' for ext in required_exts]
        missing = [ext for ext, f in zip(required_exts, files) if not f.exists()]
        if missing:
            print(f"    Chunk {chunk:2d}: MISSING {missing}")
            all_ok = False
        else:
            xtc_mb = files[0].stat().st_size / 1024 / 1024
            print(f"    Chunk {chunk:2d}: OK  ({xtc_mb:.1f} MB xtc)")

print("\n" + "=" * 60)
if all_ok:
    print("  All files verified. Drive is complete.")
    print("  Ready for Notebook 05c (local analysis).")
else:
    print("  WARNING: Some files missing — check above.")
print("=" * 60)


  pyrimethamine (5 chunks expected):
    Chunk  1: OK  (46.8 MB xtc)
    Chunk  2: OK  (46.8 MB xtc)
    Chunk  3: OK  (46.8 MB xtc)
    Chunk  4: OK  (46.8 MB xtc)
    Chunk  5: OK  (46.8 MB xtc)

  CNP0275186_1 (10 chunks expected):
    Chunk  1: OK  (46.8 MB xtc)
    Chunk  2: OK  (46.8 MB xtc)
    Chunk  3: OK  (46.8 MB xtc)
    Chunk  4: OK  (46.8 MB xtc)
    Chunk  5: OK  (46.8 MB xtc)
    Chunk  6: OK  (46.8 MB xtc)
    Chunk  7: OK  (46.8 MB xtc)
    Chunk  8: OK  (46.9 MB xtc)
    Chunk  9: OK  (46.9 MB xtc)
    Chunk 10: OK  (46.8 MB xtc)

  CNP0539885_2 (10 chunks expected):
    Chunk  1: OK  (46.8 MB xtc)
    Chunk  2: OK  (46.9 MB xtc)
    Chunk  3: OK  (46.9 MB xtc)
    Chunk  4: OK  (46.9 MB xtc)
    Chunk  5: OK  (46.9 MB xtc)
    Chunk  6: OK  (46.9 MB xtc)
    Chunk  7: OK  (46.9 MB xtc)
    Chunk  8: OK  (46.8 MB xtc)
    Chunk  9: OK  (46.9 MB xtc)
    Chunk 10: OK  (46.9 MB xtc)

  CNP0286261_0 (10 chunks expected):
    Chunk  1: OK  (46.8 MB xtc)
    Chunk  2: OK

In [ ]:
# ── Generate and save TPR files to Drive ─────────────────────────────────────
import subprocess, os, shutil, re, zipfile
from pathlib import Path

DRIVE_BASE  = Path('/content/drive/MyDrive/PfDHFR_MD')
SYSTEMS_DIR = DRIVE_BASE / 'systems'
TPR_DIR     = DRIVE_BASE / 'tpr_files'
TPR_DIR.mkdir(exist_ok=True)

compounds = ['pyrimethamine', 'CNP0275186_1', 'CNP0539885_2', 'CNP0286261_0']

os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']

for MOL_NAME in compounds:
    print(f"\n  {MOL_NAME}...")
    WORK_DIR = Path(f'/content/tpr_work/{MOL_NAME}')
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    WORK_DIR.mkdir(parents=True)
    os.chdir(WORK_DIR)

    # Extract system zip
    with zipfile.ZipFile(SYSTEMS_DIR / f'{MOL_NAME}_system.zip', 'r') as zf:
        zf.extractall(WORK_DIR)

    # Read box
    with open(WORK_DIR / 'complex.gro') as f:
        lines = f.readlines()
    box_vals = lines[-1].strip().split()
    bx, by, bz = box_vals[0], box_vals[1], box_vals[2]

    # Read mol_name from ITP
    itp_file = WORK_DIR / f'{MOL_NAME}_GMX.itp'
    mol_name_itp = None
    in_moltype = False
    for line in itp_file.read_text().splitlines():
        if '[ moleculetype ]' in line:
            in_moltype = True; continue
        if in_moltype and line.startswith(';'):
            continue
        if in_moltype and line.strip():
            mol_name_itp = line.strip().split()[0]; break

    # Patch topology
    top_file = WORK_DIR / 'topol_complex.top'
    top_text = top_file.read_text(encoding='utf-8')
    top_text = re.sub(r'"\.\./ligand_params/[^"]+\.itp"', f'"{MOL_NAME}_GMX.itp"', top_text)
    top_text = re.sub(r'#include\s+"[^"]*posre\.itp"', '#include "posre.itp"', top_text)
    top_text = re.sub(r'\bLIG\b(\s+1)', f'{mol_name_itp}\\1', top_text)
    top_file.write_text(top_text, encoding='utf-8')

    # Solvate
    subprocess.run(
        f'gmx solvate -cp complex.gro -cs spc216.gro -o solvated.gro '
        f'-p topol_complex.top -box {bx} {by} {bz}',
        shell=True, capture_output=True)
    top_text = top_file.read_text(encoding='utf-8')
    top_text = re.sub(f'({re.escape(mol_name_itp)}\\s+1)(SOL)', r'\1\nSOL', top_text)
    top_text = re.sub(f'{re.escape(mol_name_itp)}\\s+1', f'{mol_name_itp}        1', top_text)
    top_file.write_text(top_text, encoding='utf-8')

    # Ions
    subprocess.run('gmx grompp -f em.mdp -c solvated.gro -p topol_complex.top -o ions.tpr -maxwarn 20',
                   shell=True, capture_output=True)
    subprocess.run('echo "SOL" | gmx genion -s ions.tpr -o ionised.gro '
                   '-p topol_complex.top -pname NA -nname CL -neutral -conc 0.15',
                   shell=True, capture_output=True)

    # Index
    r = subprocess.run(['gmx', 'make_ndx', '-f', 'ionised.gro', '-o', 'index.ndx'],
                       input="1 | 13\nname 21 Protein_LIG\nq\n",
                       capture_output=True, text=True)
    ndx_text = (WORK_DIR / 'index.ndx').read_text()
    if 'Protein_LIG' not in ndx_text:
        (WORK_DIR / 'index.ndx').write_text(
            re.sub(r'\[ Protein_UNL \]', '[ Protein_LIG ]', ndx_text))

    # Get last GRO from Drive to use as structure for TPR
    last_chunk = 5 if MOL_NAME == 'pyrimethamine' else 10
    last_gro = DRIVE_BASE / 'md_results' / MOL_NAME / f'md_chunk{last_chunk}.gro'
    shutil.copy2(last_gro, WORK_DIR / 'last_chunk.gro')

    # Generate production TPR from last chunk GRO
    r = subprocess.run(
        f'gmx grompp -f md.mdp -c last_chunk.gro '
        f'-p topol_complex.top -n index.ndx -o md_production.tpr -maxwarn 20',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  grompp failed: {r.stderr[-300:]}")
        continue

    # Save TPR to Drive
    tpr_dest = TPR_DIR / f'{MOL_NAME}_md.tpr'
    shutil.copy2(WORK_DIR / 'md_production.tpr', tpr_dest)
    print(f"  Saved: {tpr_dest.name} ({tpr_dest.stat().st_size/1024:.0f} KB)")

print("\n" + "=" * 60)
print("  All TPR files saved to Drive/PfDHFR_MD/tpr_files/")
print("=" * 60)


  pyrimethamine...
  Saved: pyrimethamine_md.tpr (1208 KB)

  CNP0275186_1...
  Saved: CNP0275186_1_md.tpr (1219 KB)

  CNP0539885_2...
  Saved: CNP0539885_2_md.tpr (1219 KB)

  CNP0286261_0...
  Saved: CNP0286261_0_md.tpr (1216 KB)

  All TPR files saved to Drive/PfDHFR_MD/tpr_files/


In [ ]:
# ── CELL: Pre-process trajectories with gmx trjconv ──────────────────────────

import subprocess, os, shutil, time
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE   = Path('/content/drive/MyDrive/PfDHFR_MD')
RESULTS_BASE = DRIVE_BASE / 'md_results'
TPR_DIR      = DRIVE_BASE / 'tpr_files'
CLEAN_DIR    = DRIVE_BASE / 'md_results_clean'
CLEAN_DIR.mkdir(exist_ok=True)

# ── GROMACS setup — wait for full copy before proceeding ─────────────────────
print("Setting up GROMACS...")
subprocess.run('apt-get install -y -q libfftw3-dev', shell=True)

if Path('/content/gromacs_cuda').exists():
    shutil.rmtree('/content/gromacs_cuda')

os.environ['PATH'] = '/content/gromacs_cuda/bin:' + os.environ['PATH']
print("  Copying GROMACS from Drive...")
shutil.copytree(str(DRIVE_BASE / 'gromacs_cuda_binary'), '/content/gromacs_cuda')
subprocess.run('chmod -R +x /content/gromacs_cuda/bin/', shell=True)
time.sleep(15)  # wait for filesystem to flush

# Verify GROMACS works before proceeding
r = subprocess.run('gmx --version', shell=True, capture_output=True, text=True)
assert r.returncode == 0 and 'GROMACS' in r.stdout, \
    f"GROMACS not working:\n{r.stderr}"
for line in r.stdout.splitlines():
    if 'GROMACS version' in line or 'GPU support' in line:
        print(f"  {line.strip()}")
print("  GROMACS OK\n")

# ── Process each compound ─────────────────────────────────────────────────────
compounds = {
    'pyrimethamine' : 5,
    'CNP0275186_1'  : 10,
    'CNP0539885_2'  : 10,
    'CNP0286261_0'  : 10,
}

for name, n_chunks in compounds.items():
    print(f"\n{'='*60}")
    print(f"  Processing: {name}")
    print(f"{'='*60}")

    WORK_DIR = Path(f'/content/trjconv_work/{name}')
    WORK_DIR.mkdir(parents=True, exist_ok=True)

    tpr = TPR_DIR / f'{name}_md.tpr'
    assert tpr.exists(), f"TPR not found: {tpr}"
    shutil.copy2(tpr, WORK_DIR / 'md.tpr')

    # Copy XTC chunks
    chunk_files = []
    for i in range(1, n_chunks + 1):
        src = RESULTS_BASE / name / f'md_chunk{i}.xtc'
        dst = WORK_DIR / f'md_chunk{i}.xtc'
        shutil.copy2(src, dst)
        chunk_files.append(str(dst))
    print(f"  Copied {n_chunks} XTC chunks")

    # Step 1: Concatenate chunks
    concat_xtc = WORK_DIR / 'md_full.xtc'
    r = subprocess.run(
        f'gmx trjcat -f {" ".join(chunk_files)} -o {str(concat_xtc)} -cat',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  trjcat FAILED:\n{r.stderr[-400:]}")
        continue
    print(f"  md_full.xtc: {concat_xtc.stat().st_size/1024/1024:.0f} MB")

    # Step 2: Center + unwrap PBC
    clean_xtc = WORK_DIR / 'md_clean.xtc'
    r = subprocess.run(
        f'echo "1 0" | gmx trjconv '
        f'-s {WORK_DIR}/md.tpr '
        f'-f {str(concat_xtc)} '
        f'-o {str(clean_xtc)} '
        f'-center -pbc mol -ur compact',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  trjconv FAILED:\n{r.stderr[-400:]}")
        continue
    print(f"  md_clean.xtc: {clean_xtc.stat().st_size/1024/1024:.0f} MB")

    # Step 3: Save to Drive
    out_dir = CLEAN_DIR / name
    out_dir.mkdir(exist_ok=True)
    shutil.copy2(clean_xtc, out_dir / 'md_clean.xtc')
    shutil.copy2(WORK_DIR / 'md.tpr', out_dir / 'md.tpr')
    print(f"  Saved to Drive: md_clean.xtc + md.tpr")

print("\n" + "=" * 60)
print("  All clean trajectories saved to:")
print("  Drive/PfDHFR_MD/md_results_clean/")
print("  Download each compound folder and replace local trajectories")
print("=" * 60)

Mounted at /content/drive
Setting up GROMACS...
  Copying GROMACS from Drive...
  GROMACS version:    2023.3
  GPU support:        CUDA
  GROMACS OK


  Processing: pyrimethamine
  Copied 5 XTC chunks
  md_full.xtc: 234 MB
  md_clean.xtc: 232 MB
  Saved to Drive: md_clean.xtc + md.tpr

  Processing: CNP0275186_1
  Copied 10 XTC chunks
  md_full.xtc: 468 MB
  md_clean.xtc: 464 MB
  Saved to Drive: md_clean.xtc + md.tpr

  Processing: CNP0539885_2
  Copied 10 XTC chunks
  md_full.xtc: 469 MB
  md_clean.xtc: 464 MB
  Saved to Drive: md_clean.xtc + md.tpr

  Processing: CNP0286261_0
  Copied 10 XTC chunks
  md_full.xtc: 468 MB
  md_clean.xtc: 464 MB
  Saved to Drive: md_clean.xtc + md.tpr

  All clean trajectories saved to:
  Drive/PfDHFR_MD/md_results_clean/
  Download each compound folder and replace local trajectories
